# OCR / 이미지 전처리 실험 — PR 검토 안내

**목적:** 동일 표본 CPU 비교로 OCR 엔진·해상도·배치·검출 임계값을 검토하고, 고정 30장에서 현재 최선의 잠정 baseline을 확인한다. 최종 후보는 완성 모델이 아니며 날짜 해석·소비기한 판별은 범위에 포함하지 않는다.

**환경:** 프로젝트 `.venv`의 설치된 PaddleOCR/PaddleX, 로컬 `weights/` 모델, CPU threads=4. PP-OCRv5_mobile_det + korean_PP-OCRv5_mobile_rec, 입력 긴 변 512px, batch6, predict 임계값 0.7. EXIF 보정/RGB/LANCZOS, MKLDNN 활성화, 방향·왜곡 보정 비활성화.

**방법:** 기존 동일 10장 결과를 재사용하고 seed=42의 고정 30장으로 검증했다. 최종 0.7 단독 집계는 기존 단독 10장 + 저장된 0.7 재시도 10장 + 미측정 10장만 추가 실행한 결과다. 완료한 실험은 다시 실행하지 않았다. 아래 실행 셀은 기록 보존용이므로 전체 실행하지 않는다. 모델 초기화는 추론 시간에서 제외한다.

| 최종 30장 지표 | 결과 |
|---|---:|
| 숫자형 날짜 토큰 재현율 | 22/29 = 75.86% |
| OCR 문자 인식 기준, 영문 월 사례 수동 검토 포함 | 23/29 = 79.31% |
| parser/evaluator 미지원 | 002728.jpg: `04 NOV 2021` |
| OCR 실패 | 6장 |
| 평균 / 중앙 처리시간 | 1.317 / 1.020초/장 |
| 검출 박스 합계 / 평균 | 379 / 12.63 |
| 3,352장 예상시간 | 73.56분 |

완전한 라벨은 29장이다. 23/29는 새 파서나 계산 변경 결과가 아니라 숫자형 평가 성공 22장에 올바르게 인식된 영문 월 1장을 구분한 수동 검토 수치다. 002728.jpg의 영문 월 해석은 이수민 인계사항이다.

**결론:** 512px / batch6 / threshold=0.7을 잠정 baseline으로 유지한다. 정확도와 **40분 실행 제한**을 충족하지 못한다. 시간은 기존 단독·warm fallback·신규 측정을 합산한 것으로 균일한 재측정 벤치마크가 아니다. 실패 6장에는 향후 검증된 fallback이 필요하다. 추가 최적화와 50장/300장 실행은 이번 정리에서 수행하지 않았다.

상세 비교·실패 유형·인계 형식: [실험 요약](../docs/ocr_preprocess_experiment_summary.md). 데이터·라벨·모델은 Git에서 제외된다. 아래에 EasyOCR/PaddleOCR 비교와 핵심 실험의 계산 코드·결과를 보존했다. 다운로드 로그·반복 경고·오류 스택과 긴 중간 출력만 정리했다.


# EasyOCR vs PaddleOCR: CPU baseline comparison

이 노트북은 최종 제출 코드를 만드는 단계가 아니라, 동일한 validation 이미지에서 두 OCR 엔진을 공정하게 비교하기 위한 첫 실험입니다.

- 대상: `labels/labels_300.csv`에 기록된 동일한 300장
- 공통 입력: EXIF 방향 보정 → RGB 변환 → 긴 변 최대 1600px 축소
- 실행 장치: CPU only
- 비교 항목: 날짜 문자열 회수율(스크리닝 지표), 이미지당 추론 시간, 실패 이미지
- 출력 위치: `data/validation/ocr_comparison/` (Git에서 제외됨)

날짜 선택·해석 규칙의 최종 정확도가 아니라 **OCR이 정답 날짜를 읽어냈는지**를 비교합니다.


## 1. 실험 원칙

| 조건 | EasyOCR | PaddleOCR |
|---|---|---|
| 평가 이미지 | 동일한 정렬 순서 | 동일한 정렬 순서 |
| 언어 | 한국어 + 영어 | 한국어 |
| 장치 | CPU | CPU |
| 전처리 | 동일 | 동일 |
| 시간 측정 | 모델 초기화와 이미지 추론 분리 | 모델 초기화와 이미지 추론 분리 |
| 결과 보존 | 원문·신뢰도·처리시간 | 원문·신뢰도·처리시간 |

기본값은 10장 smoke test입니다. 두 엔진이 정상 작동한 뒤 `RUN_LIMIT = None`으로 바꾸면 300장 전체를 실행합니다.


In [1]:
from pathlib import Path
import gc
import json
import os
import platform
import re
import sys
import time

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

print("Python:", sys.version.split()[0])
print("OS:", platform.platform())
print("CPU count:", os.cpu_count())


Python: 3.10.11
OS: Windows-10-10.0.26200-SP0
CPU count: 8


In [2]:
# 프로젝트 루트 또는 notebooks 폴더에서 실행할 수 있습니다.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "predict.ipynb").is_file() and (p / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("itda-ocr 프로젝트 루트 또는 notebooks 폴더에서 실행하세요.")

LABEL_PATH = PROJECT_ROOT / "labels" / "labels_300.csv"
IMAGE_DIR = PROJECT_ROOT / "data" / "validation" / "label_images"
OUTPUT_DIR = PROJECT_ROOT / "data" / "validation" / "ocr_comparison"
WEIGHT_DIR = PROJECT_ROOT / "weights"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)

if not LABEL_PATH.is_file():
    raise FileNotFoundError(f"라벨 파일을 찾을 수 없습니다: {LABEL_PATH}")
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"이미지 폴더를 찾을 수 없습니다: {IMAGE_DIR}")

labels = pd.read_csv(LABEL_PATH, encoding="utf-8-sig", dtype=str).fillna("")
required = {"file_name", "image_id", "year", "month", "day", "final_date"}
missing_columns = required - set(labels.columns)
if missing_columns:
    raise ValueError(f"라벨 CSV에 필요한 열이 없습니다: {sorted(missing_columns)}")

image_by_name = {
    p.name: p for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
}
label_names = labels["file_name"].tolist()
missing_images = sorted(set(label_names) - set(image_by_name))
extra_images = sorted(set(image_by_name) - set(label_names))
if missing_images or extra_images:
    raise ValueError(
        f"CSV와 이미지 파일명이 일치하지 않습니다. missing={missing_images[:5]}, extra={extra_images[:5]}"
    )
if len(labels) != 300:
    raise ValueError(f"라벨은 300행이어야 합니다: 현재 {len(labels)}행")

labels = labels.sort_values("image_id", key=lambda s: pd.to_numeric(s, errors="coerce")).reset_index(drop=True)
labels["image_path"] = labels["file_name"].map(image_by_name)
print(f"검증 완료: 라벨 {len(labels)}행, 이미지 {len(image_by_name)}장")


검증 완료: 라벨 300행, 이미지 300장


## 2. 실행 설정

- 첫 실행은 인터넷 연결 상태에서 모델을 한 번 내려받습니다.
- 모델은 Git에서 제외되는 `weights/` 아래에 저장합니다.
- 오프라인 검사는 첫 실행이 성공한 뒤 `OFFLINE_CHECK = True`로 바꾸고, 인터넷을 끈 상태에서 커널을 재시작하여 실행합니다.


In [3]:
RUN_LIMIT = 10       # 정상 작동 확인 후 None으로 바꾸면 300장 전체 실행
MAX_LONG_SIDE = 1600
CPU_THREADS = min(4, os.cpu_count() or 1)
OFFLINE_CHECK = False

EASYOCR_MODEL_DIR = WEIGHT_DIR / "easyocr"
PADDLE_CACHE_DIR = WEIGHT_DIR / "paddlex"
EASYOCR_MODEL_DIR.mkdir(parents=True, exist_ok=True)
PADDLE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# PaddleOCR/PaddleX가 모델을 프로젝트의 ignored weights 폴더에 저장하도록 import 전에 지정합니다.
os.environ["PADDLE_PDX_CACHE_HOME"] = str(PADDLE_CACHE_DIR)
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "1" if OFFLINE_CHECK else "0"
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)

run_labels = labels.iloc[:RUN_LIMIT].copy() if RUN_LIMIT else labels.copy()
print(f"이번 실행 대상: {len(run_labels)}장 / CPU threads: {CPU_THREADS}")


이번 실행 대상: 10장 / CPU threads: 4


In [4]:
def load_common_input(path: Path, max_long_side: int = MAX_LONG_SIDE) -> np.ndarray:
    """두 엔진에 동일하게 전달할 EXIF 보정 RGB uint8 배열을 만듭니다."""
    with Image.open(path) as src:
        image = ImageOps.exif_transpose(src).convert("RGB")
    long_side = max(image.size)
    if long_side > max_long_side:
        scale = max_long_side / long_side
        new_size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        image = image.resize(new_size, Image.Resampling.LANCZOS)
    return np.asarray(image)


def normalize_digits(value) -> str:
    return re.sub(r"[^0-9]", "", str(value))


def expected_date_variants(row) -> set[str]:
    """완전한 Y/M/D 라벨을 흔한 YMD, DMY, MDY 숫자열 후보로 변환합니다."""
    if str(row["final_date"]).strip().upper() in {"", "NONE", "NAN"}:
        return set()
    try:
        year, month, day = int(row["year"]), int(row["month"]), int(row["day"])
    except (TypeError, ValueError):
        return set()
    y4, y2 = f"{year:04d}", f"{year % 100:02d}"
    mm, dd = f"{month:02d}", f"{day:02d}"
    m, d = str(month), str(day)
    return {
        y4 + mm + dd, y2 + mm + dd, y4 + m + d, y2 + m + d,
        dd + mm + y4, dd + mm + y2, d + m + y4, d + m + y2,
        mm + dd + y4, mm + dd + y2, m + d + y4, m + d + y2,
    }


def date_recalled(text: str, row):
    variants = expected_date_variants(row)
    if not variants:
        return pd.NA
    digits = normalize_digits(text)
    return any(candidate in digits for candidate in variants)


# 입력이 두 엔진에서 정말 같게 쓰이는지 한 장을 먼저 확인합니다.
sample_array = load_common_input(run_labels.iloc[0]["image_path"])
print("공통 입력 shape/dtype:", sample_array.shape, sample_array.dtype)


공통 입력 shape/dtype: (1600, 1200, 3) uint8


## 3. 엔진 초기화

모델 초기화 시간은 이미지 추론 시간과 분리해 기록합니다. EasyOCR과 PaddleOCR을 동시에 메모리에 유지하지 않고 순서대로 실행하여 CPU·메모리 조건의 영향을 줄입니다.


In [6]:
def init_easyocr():
    import easyocr

    started = time.perf_counter()
    reader = easyocr.Reader(
        ["ko", "en"],
        gpu=False,
        model_storage_directory=str(EASYOCR_MODEL_DIR),
        download_enabled=not OFFLINE_CHECK,
        verbose=True,
    )
    return reader, time.perf_counter() - started


def infer_easyocr(reader, image_array):
    raw = reader.readtext(image_array, detail=1, paragraph=False)
    detections = []
    for box, text, confidence in raw:
        detections.append({
            "text": str(text),
            "confidence": float(confidence),
            "box": np.asarray(box).tolist(),
        })
    return detections


In [5]:
def init_paddleocr():
    from paddleocr import PaddleOCR

    started = time.perf_counter()
    engine = PaddleOCR(
        lang="korean",
        text_detection_model_name="PP-OCRv5_mobile_det",
        device="cpu",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        enable_mkldnn=True,
        cpu_threads=CPU_THREADS,
    )
    return engine, time.perf_counter() - started


def _paddle_payload(item):
    payload = getattr(item, "json", item)
    if callable(payload):
        payload = payload()
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    return payload if isinstance(payload, dict) else {}


def infer_paddleocr(engine, image_array):
    predictions = list(engine.predict(image_array))
    detections = []
    for item in predictions:
        payload = _paddle_payload(item)
        texts = payload.get("rec_texts", []) or []
        scores = payload.get("rec_scores", []) or []
        boxes = payload.get("rec_polys", payload.get("dt_polys", [])) or []
        for index, text in enumerate(texts):
            confidence = float(scores[index]) if index < len(scores) else None
            box = np.asarray(boxes[index]).tolist() if index < len(boxes) else None
            detections.append({"text": str(text), "confidence": confidence, "box": box})
    return detections


In [10]:
def run_engine(engine_name, initializer, inferencer, frame):
    print(f"[{engine_name}] 모델 초기화 시작")
    engine, init_seconds = initializer()
    print(f"[{engine_name}] 초기화 완료: {init_seconds:.2f}초")

    rows = []
    for position, (_, label_row) in enumerate(frame.iterrows(), start=1):
        image_array = load_common_input(label_row["image_path"])
        started = time.perf_counter()
        detections = inferencer(engine, image_array)
        inference_seconds = time.perf_counter() - started
        detected_text = " | ".join(item["text"] for item in detections)
        confidences = [item["confidence"] for item in detections if item["confidence"] is not None]
        rows.append({
            "engine": engine_name,
            "file_name": label_row["file_name"],
            "image_id": label_row["image_id"],
            "final_date": label_row["final_date"],
            "detected_text": detected_text,
            "date_recalled": date_recalled(detected_text, label_row),
            "mean_confidence": float(np.mean(confidences)) if confidences else np.nan,
            "detection_count": len(detections),
            "inference_seconds": inference_seconds,
            "model_init_seconds": init_seconds,
            "input_height": image_array.shape[0],
            "input_width": image_array.shape[1],
            "raw_detections_json": json.dumps(detections, ensure_ascii=False),
        })
        print(f"[{engine_name}] {position:>3}/{len(frame)} {label_row['file_name']} {inference_seconds:.2f}s")

    del engine
    gc.collect()
    return pd.DataFrame(rows)


def summarize(result):
    scored = result["date_recalled"].notna()
    return {
        "engine": result["engine"].iloc[0],
        "images": len(result),
        "scored_complete_dates": int(scored.sum()),
        "date_token_recall": float(result.loc[scored, "date_recalled"].astype(bool).mean()) if scored.any() else np.nan,
        "mean_seconds_per_image": float(result["inference_seconds"].mean()),
        "median_seconds_per_image": float(result["inference_seconds"].median()),
        "estimated_3352_minutes": float(result["inference_seconds"].mean() * 3352 / 60),
        "model_init_seconds": float(result["model_init_seconds"].iloc[0]),
    }


## 4. EasyOCR 실행

처음에는 10장만 실행합니다. 모델 다운로드 때문에 첫 초기화는 시간이 걸릴 수 있습니다.


In [8]:
easy_result = run_engine("easyocr", init_easyocr, infer_easyocr, run_labels)
easy_path = OUTPUT_DIR / "easyocr_results.csv"
easy_result.to_csv(easy_path, index=False, encoding="utf-8-sig")
pd.DataFrame([summarize(easy_result)])


[easyocr] 모델 초기화 시작


[easyocr] 초기화 완료: 3.04초


,engine,images,scored_complete_dates,date_token_recall,mean_seconds_per_image,median_seconds_per_image,estimated_3352_minutes,model_init_seconds
0,easyocr,10,10,0.8,21.417346,18.841194,1196.51573,3.044631


## 5. PaddleOCR 실행

EasyOCR과 정확히 같은 이미지 배열과 순서로 실행합니다.


In [9]:
paddle_result = run_engine("paddleocr", init_paddleocr, infer_paddleocr, run_labels)
paddle_path = OUTPUT_DIR / "paddleocr_results.csv"
paddle_result.to_csv(paddle_path, index=False, encoding="utf-8-sig")
pd.DataFrame([summarize(paddle_result)])


[paddleocr] 모델 초기화 시작


[paddleocr] 초기화 완료: 4.13초


,engine,images,scored_complete_dates,date_token_recall,mean_seconds_per_image,median_seconds_per_image,estimated_3352_minutes,model_init_seconds
0,paddleocr,10,10,1.0,42.92648,39.216542,2398.159362,4.125556


## 6. 비교 결과

`date_token_recall`은 최종 소비기한 판별 정확도가 아닙니다. 라벨의 완전한 날짜가 OCR 전체 텍스트 안에 흔한 YMD/DMY/MDY 숫자 순서 중 하나로 나타났는지 확인하는 **OCR 스크리닝 지표**입니다. 최종 날짜 선택과 해석은 별도의 date-parser 평가에서 수행해야 합니다.

기존 10장 결과에서 PaddleOCR은 날짜 토큰 재현율 `1.0`으로 EasyOCR `0.8`보다 높았지만, 평균 처리시간은 `31.88초/장`으로 EasyOCR `13.69초/장`보다 약 `2.3배` 느렸습니다. 따라서 PaddleOCR이 정확도와 속도 모두 우세하다는 결론은 내리지 않습니다.

이번 속도 개선 실험의 목표 처리시간은 **0.7초/장 이하**입니다. 해상도 축소만으로 목표와 큰 차이가 남으면 반복적인 축소 실험을 중단하고, 후보 영역 탐색·날짜 후보 크롭·배치 처리 같은 구조 변경을 검토합니다.


In [10]:
combined = pd.concat([easy_result, paddle_result], ignore_index=True)
summary = pd.DataFrame([summarize(easy_result), summarize(paddle_result)])
combined.to_csv(OUTPUT_DIR / "ocr_engine_results.csv", index=False, encoding="utf-8-sig")
summary.to_csv(OUTPUT_DIR / "ocr_engine_summary.csv", index=False, encoding="utf-8-sig")
summary


,engine,images,scored_complete_dates,date_token_recall,mean_seconds_per_image,median_seconds_per_image,estimated_3352_minutes,model_init_seconds
0,easyocr,10,10,0.8,21.417346,18.841194,1196.515730,3.044631
1,paddleocr,10,10,1.0,42.926480,39.216542,2398.159362,4.125556


In [11]:
# 한 엔진만 맞힌 이미지를 우선 검토하면 차이를 빠르게 파악할 수 있습니다.
pivot = combined.pivot(index="file_name", columns="engine", values="date_recalled")
disagreements = pivot[
    pivot.notna().all(axis=1) & (pivot["easyocr"] != pivot["paddleocr"])
].copy()
disagreements.to_csv(OUTPUT_DIR / "engine_disagreements.csv", encoding="utf-8-sig")
print(f"엔진 간 성공/실패가 다른 이미지: {len(disagreements)}장")
disagreements.head(20)


엔진 간 성공/실패가 다른 이미지: 2장


engine,easyocr,paddleocr
file_name,,
000018.jpg,False,True
000060.jpg,False,True


## 7. 다음 실행 순서

1. 현재 `RUN_LIMIT = 10`, `OFFLINE_CHECK = False`로 두 엔진이 모두 실행되는지 확인합니다.
2. 정상이면 `RUN_LIMIT = None`으로 바꾸고 300장 전체 결과를 생성합니다.
3. 모델 다운로드가 끝난 뒤 커널을 재시작하고 `OFFLINE_CHECK = True`로 바꿉니다.
4. 인터넷 연결을 끈 상태에서도 10장이 실행되면 offline 실행 가능으로 기록합니다.
5. `engine_disagreements.csv`와 두 엔진 모두 실패한 이미지를 확인한 뒤에만 전처리 실험을 설계합니다.

이 단계에서는 최종 제출 파일이나 공용 `requirements.txt`를 수정하지 않습니다.


In [9]:
# Short speed trial: keep the same 10 images and alter only scale/detector limits.
SPEED_MAX_LONG_SIDE = 1000
SPEED_PADDLE_DET_LIMIT = 640
SPEED_OUTPUT_PREFIX = OUTPUT_DIR / "speed_experiment"


def load_speed_input(path: Path) -> np.ndarray:
    return load_common_input(path, max_long_side=SPEED_MAX_LONG_SIDE)


def init_easyocr_speed():
    import easyocr

    started = time.perf_counter()
    reader = easyocr.Reader(
        ["ko", "en"],
        gpu=False,
        model_storage_directory=str(EASYOCR_MODEL_DIR),
        download_enabled=not OFFLINE_CHECK,
        verbose=True,
    )
    return reader, time.perf_counter() - started


def init_paddleocr_speed():
    from paddleocr import PaddleOCR

    started = time.perf_counter()
    engine = PaddleOCR(
        lang="korean",
        text_detection_model_name="PP-OCRv5_mobile_det",
        text_det_limit_side_len=SPEED_PADDLE_DET_LIMIT,
        text_det_limit_type="max",
        device="cpu",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        enable_mkldnn=True,
        cpu_threads=CPU_THREADS,
    )
    return engine, time.perf_counter() - started


def run_speed_engine(engine_name, initializer, inferencer, frame):
    print(f"[{engine_name}] 속도 실험 모델 초기화 시작")
    engine, init_seconds = initializer()
    print(f"[{engine_name}] 초기화 완료: {init_seconds:.2f}초")

    rows = []
    for position, (_, label_row) in enumerate(frame.iterrows(), start=1):
        image_array = load_speed_input(label_row["image_path"])
        started = time.perf_counter()
        detections = inferencer(engine, image_array)
        inference_seconds = time.perf_counter() - started
        detected_text = " | ".join(item["text"] for item in detections)
        confidences = [item["confidence"] for item in detections if item["confidence"] is not None]
        rows.append({
            "engine": engine_name,
            "file_name": label_row["file_name"],
            "image_id": label_row["image_id"],
            "final_date": label_row["final_date"],
            "detected_text": detected_text,
            "date_recalled": date_recalled(detected_text, label_row),
            "mean_confidence": float(np.mean(confidences)) if confidences else np.nan,
            "detection_count": len(detections),
            "inference_seconds": inference_seconds,
            "model_init_seconds": init_seconds,
            "input_height": image_array.shape[0],
            "input_width": image_array.shape[1],
            "raw_detections_json": json.dumps(detections, ensure_ascii=False),
        })
        print(f"[{engine_name}] {position:>3}/{len(frame)} {label_row['file_name']} {inference_seconds:.2f}s")

    del engine
    gc.collect()
    return pd.DataFrame(rows)


speed_easy_result = run_speed_engine("easyocr_speed_1000", init_easyocr_speed, infer_easyocr, run_labels)
speed_easy_result.to_csv(f"{SPEED_OUTPUT_PREFIX}_easyocr_results.csv", index=False, encoding="utf-8-sig")
speed_easy_summary = summarize(speed_easy_result)

speed_paddle_result = run_speed_engine("paddleocr_speed_1000_det640", init_paddleocr_speed, infer_paddleocr, run_labels)
speed_paddle_result.to_csv(f"{SPEED_OUTPUT_PREFIX}_paddleocr_results.csv", index=False, encoding="utf-8-sig")
speed_paddle_summary = summarize(speed_paddle_result)

speed_summary = pd.DataFrame([speed_easy_summary, speed_paddle_summary])
speed_summary["max_long_side"] = SPEED_MAX_LONG_SIDE
speed_summary["paddle_det_limit_side_len"] = SPEED_PADDLE_DET_LIMIT
speed_summary.to_csv(f"{SPEED_OUTPUT_PREFIX}_summary.csv", index=False, encoding="utf-8-sig")
speed_summary


[easyocr_speed_1000] 속도 실험 모델 초기화 시작


[easyocr_speed_1000] 초기화 완료: 3.35초


In [11]:
# Continue the speed trial after EasyOCR completed and its result was saved.
speed_easy_summary = summarize(speed_easy_result)
speed_paddle_result = run_speed_engine(
    "paddleocr_speed_1000_det640",
    init_paddleocr_speed,
    infer_paddleocr,
    run_labels,
)
speed_paddle_result.to_csv(
    f"{SPEED_OUTPUT_PREFIX}_paddleocr_results.csv",
    index=False,
    encoding="utf-8-sig",
)
speed_paddle_summary = summarize(speed_paddle_result)

speed_summary = pd.DataFrame([speed_easy_summary, speed_paddle_summary])
speed_summary["max_long_side"] = SPEED_MAX_LONG_SIDE
speed_summary["paddle_det_limit_side_len"] = SPEED_PADDLE_DET_LIMIT
speed_summary.to_csv(
    f"{SPEED_OUTPUT_PREFIX}_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
speed_summary


[paddleocr_speed_1000_det640] 속도 실험 모델 초기화 시작


[paddleocr_speed_1000_det640] 초기화 완료: 4.34초


,engine,images,scored_complete_dates,date_token_recall,mean_seconds_per_image,median_seconds_per_image,estimated_3352_minutes,model_init_seconds,max_long_side,paddle_det_limit_side_len
0,easyocr_speed_1000,10,10,0.8,13.692655,12.555836,764.962998,3.345746,1000,640
1,paddleocr_speed_1000_det640,10,10,1.0,31.882973,28.049686,1781.195440,4.335125,1000,640


In [17]:
# PaddleOCR-only resolution trial on the same 10 images.
# Target: mean inference time <= 0.7 seconds per image.
PADDLE_RESOLUTION_TRIALS = (640, 512)
PADDLE_RESOLUTION_OUTPUT_PREFIX = OUTPUT_DIR / "paddle_resolution_trial"


def run_paddle_resolution_trial(max_long_side: int):
    from paddleocr import PaddleOCR

    started = time.perf_counter()
    engine = PaddleOCR(
        lang="korean",
        text_detection_model_name="PP-OCRv5_mobile_det",
        text_det_limit_side_len=max_long_side,
        text_det_limit_type="max",
        device="cpu",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        enable_mkldnn=True,
        cpu_threads=CPU_THREADS,
    )
    init_seconds = time.perf_counter() - started

    rows = []
    for position, (_, label_row) in enumerate(run_labels.iterrows(), start=1):
        image_array = load_common_input(label_row["image_path"], max_long_side=max_long_side)
        started = time.perf_counter()
        detections = infer_paddleocr(engine, image_array)
        inference_seconds = time.perf_counter() - started
        detected_text = " | ".join(item["text"] for item in detections)
        confidences = [item["confidence"] for item in detections if item["confidence"] is not None]
        rows.append({
            "engine": "paddleocr",
            "resolution": max_long_side,
            "file_name": label_row["file_name"],
            "image_id": label_row["image_id"],
            "final_date": label_row["final_date"],
            "detected_text": detected_text,
            "date_recalled": date_recalled(detected_text, label_row),
            "mean_confidence": float(np.mean(confidences)) if confidences else np.nan,
            "detection_count": len(detections),
            "inference_seconds": inference_seconds,
            "model_init_seconds": init_seconds,
            "input_height": image_array.shape[0],
            "input_width": image_array.shape[1],
            "raw_detections_json": json.dumps(detections, ensure_ascii=False),
        })
        print(f"[paddleocr {max_long_side}px] {position:>2}/{len(run_labels)} {label_row['file_name']} {inference_seconds:.2f}s")

    del engine
    gc.collect()
    result = pd.DataFrame(rows)
    result.to_csv(
        f"{PADDLE_RESOLUTION_OUTPUT_PREFIX}_{max_long_side}.csv",
        index=False,
        encoding="utf-8-sig",
    )
    scored = result["date_recalled"].notna()
    return {
        "engine": "paddleocr",
        "resolution": max_long_side,
        "images": len(result),
        "date_token_recall": float(result.loc[scored, "date_recalled"].astype(bool).mean()),
        "mean_seconds_per_image": float(result["inference_seconds"].mean()),
        "median_seconds_per_image": float(result["inference_seconds"].median()),
        "estimated_3352_minutes": float(result["inference_seconds"].mean() * 3352 / 60),
        "model_init_seconds": init_seconds,
        "target_seconds_per_image": 0.7,
        "target_met": bool(result["inference_seconds"].mean() <= 0.7),
    }


paddle_resolution_summary = pd.DataFrame(
    [run_paddle_resolution_trial(resolution) for resolution in PADDLE_RESOLUTION_TRIALS]
)
paddle_resolution_summary.to_csv(
    f"{PADDLE_RESOLUTION_OUTPUT_PREFIX}_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
paddle_resolution_summary


,engine,resolution,images,date_token_recall,mean_seconds_per_image,median_seconds_per_image,estimated_3352_minutes,model_init_seconds,target_seconds_per_image,target_met
0,paddleocr,640,10,1.0,30.834332,21.636042,1722.611360,1.848399,0.7,False
1,paddleocr,512,10,1.0,27.485081,19.230330,1535.499876,2.590031,0.7,False


In [6]:
# Corrective 512px trial: same 10 images, explicitly use the Korean mobile recognizer.
MOBILE_REC_MODEL = "korean_PP-OCRv5_mobile_rec"
MOBILE_TRIAL_RESOLUTION = 512
MOBILE_OUTPUT_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec"

from paddleocr import PaddleOCR

started = time.perf_counter()
mobile_engine = PaddleOCR(
    lang="korean",
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name=MOBILE_REC_MODEL,
    text_det_limit_side_len=MOBILE_TRIAL_RESOLUTION,
    text_det_limit_type="max",
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=True,
    cpu_threads=CPU_THREADS,
)
mobile_init_seconds = time.perf_counter() - started
print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] 초기화 완료: {mobile_init_seconds:.2f}초")

mobile_rows = []
for position, (_, label_row) in enumerate(run_labels.iterrows(), start=1):
    image_array = load_common_input(label_row["image_path"], max_long_side=MOBILE_TRIAL_RESOLUTION)
    started = time.perf_counter()
    detections = infer_paddleocr(mobile_engine, image_array)
    inference_seconds = time.perf_counter() - started
    detected_text = " | ".join(item["text"] for item in detections)
    confidences = [item["confidence"] for item in detections if item["confidence"] is not None]
    mobile_rows.append({
        "engine": "paddleocr",
        "model_name": MOBILE_REC_MODEL,
        "det_model_name": "PP-OCRv5_mobile_det",
        "resolution": MOBILE_TRIAL_RESOLUTION,
        "file_name": label_row["file_name"],
        "image_id": label_row["image_id"],
        "final_date": label_row["final_date"],
        "detected_text": detected_text,
        "date_recalled": date_recalled(detected_text, label_row),
        "mean_confidence": float(np.mean(confidences)) if confidences else np.nan,
        "detection_count": len(detections),
        "inference_seconds": inference_seconds,
        "model_init_seconds": mobile_init_seconds,
        "input_height": image_array.shape[0],
        "input_width": image_array.shape[1],
        "raw_detections_json": json.dumps(detections, ensure_ascii=False),
    })
    print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] {position:>2}/{len(run_labels)} {label_row['file_name']} {inference_seconds:.2f}s")

del mobile_engine
gc.collect()
mobile_result = pd.DataFrame(mobile_rows)
mobile_result.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_results.csv",
    index=False,
    encoding="utf-8-sig",
)

scored = mobile_result["date_recalled"].notna()
mobile_summary = {
    "engine": "paddleocr",
    "model_name": MOBILE_REC_MODEL,
    "det_model_name": "PP-OCRv5_mobile_det",
    "resolution": MOBILE_TRIAL_RESOLUTION,
    "images": len(mobile_result),
    "date_token_recall": float(mobile_result.loc[scored, "date_recalled"].astype(bool).mean()),
    "mean_seconds_per_image": float(mobile_result["inference_seconds"].mean()),
    "median_seconds_per_image": float(mobile_result["inference_seconds"].median()),
    "estimated_3352_minutes": float(mobile_result["inference_seconds"].mean() * 3352 / 60),
    "model_init_seconds": mobile_init_seconds,
    "target_seconds_per_image": 0.7,
    "target_met": bool(mobile_result["inference_seconds"].mean() <= 0.7),
}
mobile_summary_df = pd.DataFrame([mobile_summary])
mobile_summary_df.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# Compare only against the pre-existing 512px result, which used PP-OCRv6_medium_rec.
previous_summary = pd.read_csv(OUTPUT_DIR / "paddle_resolution_trial_summary.csv")
previous_512 = previous_summary.loc[previous_summary["resolution"] == MOBILE_TRIAL_RESOLUTION].iloc[0]
comparison = pd.DataFrame([
    {
        "metric": "date_token_recall",
        "PP-OCRv6_medium_rec": previous_512["date_token_recall"],
        MOBILE_REC_MODEL: mobile_summary["date_token_recall"],
        "mobile_minus_v6": mobile_summary["date_token_recall"] - previous_512["date_token_recall"],
    },
    {
        "metric": "mean_seconds_per_image",
        "PP-OCRv6_medium_rec": previous_512["mean_seconds_per_image"],
        MOBILE_REC_MODEL: mobile_summary["mean_seconds_per_image"],
        "mobile_minus_v6": mobile_summary["mean_seconds_per_image"] - previous_512["mean_seconds_per_image"],
    },
    {
        "metric": "median_seconds_per_image",
        "PP-OCRv6_medium_rec": previous_512["median_seconds_per_image"],
        MOBILE_REC_MODEL: mobile_summary["median_seconds_per_image"],
        "mobile_minus_v6": mobile_summary["median_seconds_per_image"] - previous_512["median_seconds_per_image"],
    },
    {
        "metric": "estimated_3352_minutes",
        "PP-OCRv6_medium_rec": previous_512["estimated_3352_minutes"],
        MOBILE_REC_MODEL: mobile_summary["estimated_3352_minutes"],
        "mobile_minus_v6": mobile_summary["estimated_3352_minutes"] - previous_512["estimated_3352_minutes"],
    },
])
comparison.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_vs_PP-OCRv6_medium_rec.csv",
    index=False,
    encoding="utf-8-sig",
)
print("\n모델 비교 결과")
print(comparison.to_string(index=False))
if mobile_summary["mean_seconds_per_image"] > 0.7:
    print("\n판정: 목표 0.7초/장을 크게 초과하므로 해상도 실험을 종료합니다.")
    print("제안: 전체 이미지 OCR 대신 텍스트 후보 영역 탐색 후 날짜 후보 크롭만 인식하고, 후보 크롭을 배치 처리하는 구조로 변경하세요.")
else:
    print("\n판정: 목표 0.7초/장을 달성했습니다.")


[paddleocr 512px/korean_PP-OCRv5_mobile_rec] 초기화 완료: 9.00초

모델 비교 결과
                  metric  PP-OCRv6_medium_rec  korean_PP-OCRv5_mobile_rec  mobile_minus_v6
       date_token_recall             1.000000                    1.000000         0.000000
  mean_seconds_per_image            27.485081                    1.729757       -25.755324
median_seconds_per_image            19.230330                    1.280868       -17.949462
  estimated_3352_minutes          1535.499876                   96.635766     -1438.864110

판정: 목표 0.7초/장을 크게 초과하므로 해상도 실험을 종료합니다.
제안: 전체 이미지 OCR 대신 텍스트 후보 영역 탐색 후 날짜 후보 크롭만 인식하고, 후보 크롭을 배치 처리하는 구조로 변경하세요.


In [7]:
# Batch-size trial: preserve the 512px mobile baseline and change only recognition batch size.
BATCH_SIZE = 6
BATCH_TRIAL_OUTPUT_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch6"

started = time.perf_counter()
batch_engine = PaddleOCR(
    lang="korean",
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name="korean_PP-OCRv5_mobile_rec",
    text_recognition_batch_size=BATCH_SIZE,
    text_det_limit_side_len=512,
    text_det_limit_type="max",
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=True,
    cpu_threads=CPU_THREADS,
)
batch_init_seconds = time.perf_counter() - started
print(f"[paddleocr 512px/korean_PP-OCRv5_mobile_rec batch={BATCH_SIZE}] 초기화 완료: {batch_init_seconds:.2f}초")

batch_rows = []
for position, (_, label_row) in enumerate(run_labels.iterrows(), start=1):
    image_array = load_common_input(label_row["image_path"], max_long_side=512)
    started = time.perf_counter()
    detections = infer_paddleocr(batch_engine, image_array)
    inference_seconds = time.perf_counter() - started
    detected_text = " | ".join(item["text"] for item in detections)
    confidences = [item["confidence"] for item in detections if item["confidence"] is not None]
    batch_rows.append({
        "engine": "paddleocr",
        "model_name": "korean_PP-OCRv5_mobile_rec",
        "det_model_name": "PP-OCRv5_mobile_det",
        "text_recognition_batch_size": BATCH_SIZE,
        "resolution": 512,
        "file_name": label_row["file_name"],
        "image_id": label_row["image_id"],
        "final_date": label_row["final_date"],
        "detected_text": detected_text,
        "date_recalled": date_recalled(detected_text, label_row),
        "mean_confidence": float(np.mean(confidences)) if confidences else np.nan,
        "detection_count": len(detections),
        "inference_seconds": inference_seconds,
        "model_init_seconds": batch_init_seconds,
        "input_height": image_array.shape[0],
        "input_width": image_array.shape[1],
        "raw_detections_json": json.dumps(detections, ensure_ascii=False),
    })
    print(f"[paddleocr 512px/mobile_rec batch={BATCH_SIZE}] {position:>2}/{len(run_labels)} {label_row['file_name']} {inference_seconds:.2f}s")

del batch_engine
gc.collect()
batch_result = pd.DataFrame(batch_rows)
batch_result.to_csv(
    f"{BATCH_TRIAL_OUTPUT_PREFIX}_results.csv",
    index=False,
    encoding="utf-8-sig",
)

scored = batch_result["date_recalled"].notna()
batch_summary = {
    "engine": "paddleocr",
    "model_name": "korean_PP-OCRv5_mobile_rec",
    "det_model_name": "PP-OCRv5_mobile_det",
    "text_recognition_batch_size": BATCH_SIZE,
    "resolution": 512,
    "images": len(batch_result),
    "date_token_recall": float(batch_result.loc[scored, "date_recalled"].astype(bool).mean()),
    "mean_seconds_per_image": float(batch_result["inference_seconds"].mean()),
    "median_seconds_per_image": float(batch_result["inference_seconds"].median()),
    "estimated_3352_minutes": float(batch_result["inference_seconds"].mean() * 3352 / 60),
    "model_init_seconds": batch_init_seconds,
    "target_seconds_per_image": 0.7,
    "target_met": bool(batch_result["inference_seconds"].mean() <= 0.7),
}
batch_summary_df = pd.DataFrame([batch_summary])
batch_summary_df.to_csv(
    f"{BATCH_TRIAL_OUTPUT_PREFIX}_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# Compare against the confirmed baseline without rerunning it.
baseline_summary = pd.read_csv(
    OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_summary.csv"
).iloc[0]
batch_comparison = pd.DataFrame([
    {
        "metric": "date_token_recall",
        "baseline_korean_PP-OCRv5_mobile_rec_batch_default": baseline_summary["date_token_recall"],
        "batch6_korean_PP-OCRv5_mobile_rec": batch_summary["date_token_recall"],
        "batch6_minus_baseline": batch_summary["date_token_recall"] - baseline_summary["date_token_recall"],
    },
    {
        "metric": "mean_seconds_per_image",
        "baseline_korean_PP-OCRv5_mobile_rec_batch_default": baseline_summary["mean_seconds_per_image"],
        "batch6_korean_PP-OCRv5_mobile_rec": batch_summary["mean_seconds_per_image"],
        "batch6_minus_baseline": batch_summary["mean_seconds_per_image"] - baseline_summary["mean_seconds_per_image"],
    },
    {
        "metric": "median_seconds_per_image",
        "baseline_korean_PP-OCRv5_mobile_rec_batch_default": baseline_summary["median_seconds_per_image"],
        "batch6_korean_PP-OCRv5_mobile_rec": batch_summary["median_seconds_per_image"],
        "batch6_minus_baseline": batch_summary["median_seconds_per_image"] - baseline_summary["median_seconds_per_image"],
    },
    {
        "metric": "estimated_3352_minutes",
        "baseline_korean_PP-OCRv5_mobile_rec_batch_default": baseline_summary["estimated_3352_minutes"],
        "batch6_korean_PP-OCRv5_mobile_rec": batch_summary["estimated_3352_minutes"],
        "batch6_minus_baseline": batch_summary["estimated_3352_minutes"] - baseline_summary["estimated_3352_minutes"],
    },
])
batch_comparison.to_csv(
    f"{BATCH_TRIAL_OUTPUT_PREFIX}_vs_baseline.csv",
    index=False,
    encoding="utf-8-sig",
)
print("\n기준선 대비 batch_size=6 비교")
print(batch_comparison.to_string(index=False))


[paddleocr 512px/korean_PP-OCRv5_mobile_rec batch=6] 초기화 완료: 1.84초

기준선 대비 batch_size=6 비교
                  metric  baseline_korean_PP-OCRv5_mobile_rec_batch_default  batch6_korean_PP-OCRv5_mobile_rec  batch6_minus_baseline
       date_token_recall                                           1.000000                           1.000000               0.000000
  mean_seconds_per_image                                           1.729757                           1.822303               0.092545
median_seconds_per_image                                           1.280868                           1.254615              -0.026253
  estimated_3352_minutes                                          96.635766                         101.805969               5.170204


## 8. 384px 추가 실험 — 기본 batch 512px 기준선 비교

기존 실험 셀은 재실행하지 않습니다. 저장된 512px 모바일 모델 결과 CSV에서 동일한 10장과 순서를 가져옵니다. 입력 긴 변 및 검출 해상도 한도만 512→384로 변경하고, 두 모델·기본 batch·CPU_THREADS=min(4, os.cpu_count() or 1)·EXIF/RGB/LANCZOS 전처리·날짜 토큰 회수율·추론 시간 측정은 기준선과 동일하게 유지합니다. 기존 함수 정의를 복사하여 이 셀만 독립 실행할 수 있습니다. 384px CSV가 있으면 실행을 중단합니다. 결과·요약·512px 비교를 별도 CSV로 저장합니다.


In [1]:
from pathlib import Path
import gc
import json
import os
import re
import time
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (cwd, *cwd.parents) if (p / "predict.ipynb").is_file() and (p / "data").is_dir())
OUTPUT_DIR = PROJECT_ROOT / "data/validation/ocr_comparison"
BASELINE_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec"
TRIAL_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_384_korean_PP-OCRv5_mobile_rec"
assert not list(OUTPUT_DIR.glob(TRIAL_PREFIX.name + "*.csv")), "384px result exists: do not rerun"
baseline_result = pd.read_csv(f"{BASELINE_PREFIX}_results.csv", dtype={"image_id": str})
baseline_summary = pd.read_csv(f"{BASELINE_PREFIX}_summary.csv").iloc[0]
assert len(baseline_result) == 10 and baseline_result.file_name.is_unique
assert baseline_result.resolution.eq(512).all()
assert baseline_result.model_name.eq("korean_PP-OCRv5_mobile_rec").all()
assert baseline_result.det_model_name.eq("PP-OCRv5_mobile_det").all()
labels = pd.read_csv(PROJECT_ROOT / "labels/labels_300.csv", encoding="utf-8-sig", dtype=str).fillna("")
run_labels = labels.set_index("file_name").loc[baseline_result.file_name].reset_index()
assert run_labels.image_id.tolist() == baseline_result.image_id.tolist()
assert run_labels.final_date.tolist() == baseline_result.final_date.tolist()
image_by_name = {p.name: p for p in (PROJECT_ROOT / "data/validation/label_images").rglob("*") if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
run_labels["image_path"] = run_labels.file_name.map(image_by_name)
assert len(run_labels) == 10 and run_labels.image_path.notna().all()
CPU_THREADS = min(4, os.cpu_count() or 1)
MAX_LONG_SIDE = 1600
os.environ["PADDLE_PDX_CACHE_HOME"] = str(PROJECT_ROOT / "weights/paddlex")
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "0"
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
print(f"Same baseline images: {len(run_labels)}, CPU threads: {CPU_THREADS}, batch: default")


def load_common_input(path: Path, max_long_side: int = MAX_LONG_SIDE) -> np.ndarray:
    """두 엔진에 동일하게 전달할 EXIF 보정 RGB uint8 배열을 만듭니다."""
    with Image.open(path) as src:
        image = ImageOps.exif_transpose(src).convert("RGB")
    long_side = max(image.size)
    if long_side > max_long_side:
        scale = max_long_side / long_side
        new_size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        image = image.resize(new_size, Image.Resampling.LANCZOS)
    return np.asarray(image)

def normalize_digits(value) -> str:
    return re.sub(r"[^0-9]", "", str(value))

def expected_date_variants(row) -> set[str]:
    """완전한 Y/M/D 라벨을 흔한 YMD, DMY, MDY 숫자열 후보로 변환합니다."""
    if str(row["final_date"]).strip().upper() in {"", "NONE", "NAN"}:
        return set()
    try:
        year, month, day = int(row["year"]), int(row["month"]), int(row["day"])
    except (TypeError, ValueError):
        return set()
    y4, y2 = f"{year:04d}", f"{year % 100:02d}"
    mm, dd = f"{month:02d}", f"{day:02d}"
    m, d = str(month), str(day)
    return {
        y4 + mm + dd, y2 + mm + dd, y4 + m + d, y2 + m + d,
        dd + mm + y4, dd + mm + y2, d + m + y4, d + m + y2,
        mm + dd + y4, mm + dd + y2, m + d + y4, m + d + y2,
    }

def date_recalled(text: str, row):
    variants = expected_date_variants(row)
    if not variants:
        return pd.NA
    digits = normalize_digits(text)
    return any(candidate in digits for candidate in variants)

def _paddle_payload(item):
    payload = getattr(item, "json", item)
    if callable(payload):
        payload = payload()
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    return payload if isinstance(payload, dict) else {}

def infer_paddleocr(engine, image_array):
    predictions = list(engine.predict(image_array))
    detections = []
    for item in predictions:
        payload = _paddle_payload(item)
        texts = payload.get("rec_texts", []) or []
        scores = payload.get("rec_scores", []) or []
        boxes = payload.get("rec_polys", payload.get("dt_polys", [])) or []
        for index, text in enumerate(texts):
            confidence = float(scores[index]) if index < len(scores) else None
            box = np.asarray(boxes[index]).tolist() if index < len(boxes) else None
            detections.append({"text": str(text), "confidence": confidence, "box": box})
    return detections

# 384px trial: identical mobile baseline settings except resolution.
MOBILE_REC_MODEL = "korean_PP-OCRv5_mobile_rec"
MOBILE_TRIAL_RESOLUTION = 384
MOBILE_OUTPUT_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_384_korean_PP-OCRv5_mobile_rec"

from paddleocr import PaddleOCR

started = time.perf_counter()
mobile_engine = PaddleOCR(
    lang="korean",
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name=MOBILE_REC_MODEL,
    text_det_limit_side_len=MOBILE_TRIAL_RESOLUTION,
    text_det_limit_type="max",
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=True,
    cpu_threads=CPU_THREADS,
)
mobile_init_seconds = time.perf_counter() - started
print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] 초기화 완료: {mobile_init_seconds:.2f}초")

mobile_rows = []
for position, (_, label_row) in enumerate(run_labels.iterrows(), start=1):
    image_array = load_common_input(label_row["image_path"], max_long_side=MOBILE_TRIAL_RESOLUTION)
    started = time.perf_counter()
    detections = infer_paddleocr(mobile_engine, image_array)
    inference_seconds = time.perf_counter() - started
    detected_text = " | ".join(item["text"] for item in detections)
    confidences = [item["confidence"] for item in detections if item["confidence"] is not None]
    mobile_rows.append({
        "engine": "paddleocr",
        "model_name": MOBILE_REC_MODEL,
        "det_model_name": "PP-OCRv5_mobile_det",
        "resolution": MOBILE_TRIAL_RESOLUTION,
        "file_name": label_row["file_name"],
        "image_id": label_row["image_id"],
        "final_date": label_row["final_date"],
        "detected_text": detected_text,
        "date_recalled": date_recalled(detected_text, label_row),
        "mean_confidence": float(np.mean(confidences)) if confidences else np.nan,
        "detection_count": len(detections),
        "inference_seconds": inference_seconds,
        "model_init_seconds": mobile_init_seconds,
        "input_height": image_array.shape[0],
        "input_width": image_array.shape[1],
        "raw_detections_json": json.dumps(detections, ensure_ascii=False),
    })
    print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] {position:>2}/{len(run_labels)} {label_row['file_name']} {inference_seconds:.2f}s")

del mobile_engine
gc.collect()
mobile_result = pd.DataFrame(mobile_rows)
mobile_result.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_results.csv",
    index=False,
    encoding="utf-8-sig",
)

scored = mobile_result["date_recalled"].notna()
mobile_summary = {
    "engine": "paddleocr",
    "model_name": MOBILE_REC_MODEL,
    "det_model_name": "PP-OCRv5_mobile_det",
    "resolution": MOBILE_TRIAL_RESOLUTION,
    "images": len(mobile_result),
    "date_token_recall": float(mobile_result.loc[scored, "date_recalled"].astype(bool).mean()),
    "mean_seconds_per_image": float(mobile_result["inference_seconds"].mean()),
    "median_seconds_per_image": float(mobile_result["inference_seconds"].median()),
    "estimated_3352_minutes": float(mobile_result["inference_seconds"].mean() * 3352 / 60),
    "model_init_seconds": mobile_init_seconds,
    "target_seconds_per_image": 0.7,
    "target_met": bool(mobile_result["inference_seconds"].mean() <= 0.7),
}
mobile_summary_df = pd.DataFrame([mobile_summary])
mobile_summary_df.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


assert mobile_result.file_name.tolist() == baseline_result.file_name.tolist()
metrics = ["date_token_recall", "mean_seconds_per_image", "median_seconds_per_image", "estimated_3352_minutes"]
resolution_comparison = pd.DataFrame([
    {"metric": metric, "baseline_512": baseline_summary[metric], "trial_384": mobile_summary[metric],
     "384_minus_512": mobile_summary[metric] - baseline_summary[metric]}
    for metric in metrics
])
resolution_comparison.to_csv(f"{TRIAL_PREFIX}_vs_512.csv", index=False, encoding="utf-8-sig")
print(resolution_comparison.to_string(index=False))
print("Mean time reduction (%):", 100 * (1 - mobile_summary["mean_seconds_per_image"] / baseline_summary["mean_seconds_per_image"]))
print("Recall changes:")
print(baseline_result[["file_name", "date_recalled"]].merge(mobile_result[["file_name", "date_recalled"]], on="file_name", suffixes=("_512", "_384")).to_string(index=False))


Same baseline images: 10, CPU threads: 4, batch: default


[paddleocr 384px/korean_PP-OCRv5_mobile_rec] 초기화 완료: 19.85초


metric  baseline_512  trial_384  384_minus_512
       date_token_recall      1.000000   1.000000       0.000000
  mean_seconds_per_image      1.729757   2.499261       0.769504
median_seconds_per_image      1.280868   2.107801       0.826932
  estimated_3352_minutes     96.635766 139.625365      42.989599
Mean time reduction (%): -44.48622018692219
Recall changes:
 file_name  date_recalled_512  date_recalled_384
000007.jpg               True               True
000018.jpg               True               True
000021.jpg               True               True
000026.jpg               True               True
000030.jpg               True               True
000058.jpg               True               True
000060.jpg               True               True
000075.jpg               True               True
000099.jpg               True               True
000110.jpg               True               True


In [1]:
# 512px recognition batch-size experiment: execute this new cell only.
# Saved default-batch baseline (cell 22) is interpreted as actual batch size 6.
# Keep both mobile models, CPU threads=4, preprocessing, scoring, and timing unchanged.
# Never rerun completed experiments; use only the same 10 filenames and order from CSV.
from pathlib import Path
import gc
import json
import os
import re
import time
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (cwd, *cwd.parents) if (p / "predict.ipynb").is_file() and (p / "data").is_dir())
OUTPUT_DIR = PROJECT_ROOT / "data/validation/ocr_comparison"
BASELINE_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec"
TRIAL_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch1"
assert not list(OUTPUT_DIR.glob(TRIAL_PREFIX.name + "*.csv")), "batch1 result exists: do not rerun"
baseline_result = pd.read_csv(f"{BASELINE_PREFIX}_results.csv", dtype={"image_id": str})
baseline_summary = pd.read_csv(f"{BASELINE_PREFIX}_summary.csv").iloc[0]
assert len(baseline_result) == 10 and baseline_result.file_name.is_unique
assert baseline_result.resolution.eq(512).all()
assert baseline_result.model_name.eq("korean_PP-OCRv5_mobile_rec").all()
assert baseline_result.det_model_name.eq("PP-OCRv5_mobile_det").all()
labels = pd.read_csv(PROJECT_ROOT / "labels/labels_300.csv", encoding="utf-8-sig", dtype=str).fillna("")
run_labels = labels.set_index("file_name").loc[baseline_result.file_name].reset_index()
assert run_labels.image_id.tolist() == baseline_result.image_id.tolist()
assert run_labels.final_date.tolist() == baseline_result.final_date.tolist()
image_by_name = {p.name: p for p in (PROJECT_ROOT / "data/validation/label_images").rglob("*") if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
run_labels["image_path"] = run_labels.file_name.map(image_by_name)
assert len(run_labels) == 10 and run_labels.image_path.notna().all()
CPU_THREADS = 4
BATCH_SIZE = 1
MAX_LONG_SIDE = 1600
os.environ["PADDLE_PDX_CACHE_HOME"] = str(PROJECT_ROOT / "weights/paddlex")
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "0"
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
print(f"Same baseline images: {len(run_labels)}, CPU threads: {CPU_THREADS}, batch: 1 (saved baseline: batch 6)")


def load_common_input(path: Path, max_long_side: int = MAX_LONG_SIDE) -> np.ndarray:
    """두 엔진에 동일하게 전달할 EXIF 보정 RGB uint8 배열을 만듭니다."""
    with Image.open(path) as src:
        image = ImageOps.exif_transpose(src).convert("RGB")
    long_side = max(image.size)
    if long_side > max_long_side:
        scale = max_long_side / long_side
        new_size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        image = image.resize(new_size, Image.Resampling.LANCZOS)
    return np.asarray(image)

def normalize_digits(value) -> str:
    return re.sub(r"[^0-9]", "", str(value))

def expected_date_variants(row) -> set[str]:
    """완전한 Y/M/D 라벨을 흔한 YMD, DMY, MDY 숫자열 후보로 변환합니다."""
    if str(row["final_date"]).strip().upper() in {"", "NONE", "NAN"}:
        return set()
    try:
        year, month, day = int(row["year"]), int(row["month"]), int(row["day"])
    except (TypeError, ValueError):
        return set()
    y4, y2 = f"{year:04d}", f"{year % 100:02d}"
    mm, dd = f"{month:02d}", f"{day:02d}"
    m, d = str(month), str(day)
    return {
        y4 + mm + dd, y2 + mm + dd, y4 + m + d, y2 + m + d,
        dd + mm + y4, dd + mm + y2, d + m + y4, d + m + y2,
        mm + dd + y4, mm + dd + y2, m + d + y4, m + d + y2,
    }

def date_recalled(text: str, row):
    variants = expected_date_variants(row)
    if not variants:
        return pd.NA
    digits = normalize_digits(text)
    return any(candidate in digits for candidate in variants)

def _paddle_payload(item):
    payload = getattr(item, "json", item)
    if callable(payload):
        payload = payload()
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    return payload if isinstance(payload, dict) else {}

def infer_paddleocr(engine, image_array):
    predictions = list(engine.predict(image_array))
    detections = []
    for item in predictions:
        payload = _paddle_payload(item)
        texts = payload.get("rec_texts", []) or []
        scores = payload.get("rec_scores", []) or []
        boxes = payload.get("rec_polys", payload.get("dt_polys", [])) or []
        for index, text in enumerate(texts):
            confidence = float(scores[index]) if index < len(scores) else None
            box = np.asarray(boxes[index]).tolist() if index < len(boxes) else None
            detections.append({"text": str(text), "confidence": confidence, "box": box})
    return detections

# Batch=1 trial: identical 512px mobile baseline settings except recognition batch size.
MOBILE_REC_MODEL = "korean_PP-OCRv5_mobile_rec"
MOBILE_TRIAL_RESOLUTION = 512
MOBILE_OUTPUT_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch1"

from paddleocr import PaddleOCR

started = time.perf_counter()
mobile_engine = PaddleOCR(
    lang="korean",
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name=MOBILE_REC_MODEL,
    text_recognition_batch_size=BATCH_SIZE,
    text_det_limit_side_len=MOBILE_TRIAL_RESOLUTION,
    text_det_limit_type="max",
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=True,
    cpu_threads=CPU_THREADS,
)
mobile_init_seconds = time.perf_counter() - started
print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] 초기화 완료: {mobile_init_seconds:.2f}초")

mobile_rows = []
for position, (_, label_row) in enumerate(run_labels.iterrows(), start=1):
    image_array = load_common_input(label_row["image_path"], max_long_side=MOBILE_TRIAL_RESOLUTION)
    started = time.perf_counter()
    detections = infer_paddleocr(mobile_engine, image_array)
    inference_seconds = time.perf_counter() - started
    detected_text = " | ".join(item["text"] for item in detections)
    confidences = [item["confidence"] for item in detections if item["confidence"] is not None]
    mobile_rows.append({
        "engine": "paddleocr",
        "model_name": MOBILE_REC_MODEL,
        "det_model_name": "PP-OCRv5_mobile_det",
        "resolution": MOBILE_TRIAL_RESOLUTION,
        "text_recognition_batch_size": BATCH_SIZE,
        "cpu_threads": CPU_THREADS,
        "file_name": label_row["file_name"],
        "image_id": label_row["image_id"],
        "final_date": label_row["final_date"],
        "detected_text": detected_text,
        "date_recalled": date_recalled(detected_text, label_row),
        "mean_confidence": float(np.mean(confidences)) if confidences else np.nan,
        "detection_count": len(detections),
        "inference_seconds": inference_seconds,
        "model_init_seconds": mobile_init_seconds,
        "input_height": image_array.shape[0],
        "input_width": image_array.shape[1],
        "raw_detections_json": json.dumps(detections, ensure_ascii=False),
    })
    print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] {position:>2}/{len(run_labels)} {label_row['file_name']} {inference_seconds:.2f}s")

del mobile_engine
gc.collect()
mobile_result = pd.DataFrame(mobile_rows)
mobile_result.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_results.csv",
    index=False,
    encoding="utf-8-sig",
    mode="x",
)

scored = mobile_result["date_recalled"].notna()
mobile_summary = {
    "engine": "paddleocr",
    "model_name": MOBILE_REC_MODEL,
    "det_model_name": "PP-OCRv5_mobile_det",
    "resolution": MOBILE_TRIAL_RESOLUTION,
        "text_recognition_batch_size": BATCH_SIZE,
        "cpu_threads": CPU_THREADS,
    "images": len(mobile_result),
    "date_token_recall": float(mobile_result.loc[scored, "date_recalled"].astype(bool).mean()),
    "mean_seconds_per_image": float(mobile_result["inference_seconds"].mean()),
    "median_seconds_per_image": float(mobile_result["inference_seconds"].median()),
    "estimated_3352_minutes": float(mobile_result["inference_seconds"].mean() * 3352 / 60),
    "model_init_seconds": mobile_init_seconds,
    "target_seconds_per_image": 0.7,
    "target_met": bool(mobile_result["inference_seconds"].mean() <= 0.7),
}
mobile_summary_df = pd.DataFrame([mobile_summary])
mobile_summary_df.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_summary.csv",
    index=False,
    encoding="utf-8-sig",
    mode="x",
)


assert mobile_result.file_name.tolist() == baseline_result.file_name.tolist()
metrics = ["date_token_recall", "mean_seconds_per_image", "median_seconds_per_image", "estimated_3352_minutes"]
batch_comparison = pd.DataFrame([
    {"metric": metric, "baseline_batch6": baseline_summary[metric], "trial_batch1": mobile_summary[metric],
     "batch1_minus_batch6": mobile_summary[metric] - baseline_summary[metric]}
    for metric in metrics
])
batch_comparison.to_csv(f"{TRIAL_PREFIX}_vs_batch6.csv", index=False, encoding="utf-8-sig", mode="x")
print(batch_comparison.to_string(index=False))
print("Mean time reduction (%):", 100 * (1 - mobile_summary["mean_seconds_per_image"] / baseline_summary["mean_seconds_per_image"]))
print("Recall changes:")
print(baseline_result[["file_name", "date_recalled"]].merge(mobile_result[["file_name", "date_recalled"]], on="file_name", suffixes=("_batch6", "_batch1")).to_string(index=False))


Same baseline images: 10, CPU threads: 4, batch: 1 (saved baseline: batch 6)


[paddleocr 512px/korean_PP-OCRv5_mobile_rec] 초기화 완료: 12.63초


metric  baseline_batch6  trial_batch1  batch1_minus_batch6
       date_token_recall         1.000000      1.000000             0.000000
  mean_seconds_per_image         1.729757      3.286808             1.557051
median_seconds_per_image         1.280868      2.052557             0.771689
  estimated_3352_minutes        96.635766    183.623011            86.987246
Mean time reduction (%): -90.01558103123135
Recall changes:
 file_name  date_recalled_batch6  date_recalled_batch1
000007.jpg                  True                  True
000018.jpg                  True                  True
000021.jpg                  True                  True
000026.jpg                  True                  True
000030.jpg                  True                  True
000058.jpg                  True                  True
000060.jpg                  True                  True
000075.jpg                  True                  True
000099.jpg                  True                  True
000110.jpg             

In [1]:
# 512px recognition batch-size experiment: execute this new cell only.
# Saved default-batch baseline (cell 22) is interpreted as actual batch size 6.
# Keep both mobile models, CPU threads=4, preprocessing, scoring, and timing unchanged.
# Never rerun completed experiments; use only the same 10 filenames and order from CSV.
from pathlib import Path
import gc
import json
import os
import re
import time
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (cwd, *cwd.parents) if (p / "predict.ipynb").is_file() and (p / "data").is_dir())
OUTPUT_DIR = PROJECT_ROOT / "data/validation/ocr_comparison"
BASELINE_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec"
TRIAL_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch12"
assert not list(OUTPUT_DIR.glob(TRIAL_PREFIX.name + "*.csv")), "batch12 result exists: do not rerun"
baseline_result = pd.read_csv(f"{BASELINE_PREFIX}_results.csv", dtype={"image_id": str})
baseline_summary = pd.read_csv(f"{BASELINE_PREFIX}_summary.csv").iloc[0]
assert len(baseline_result) == 10 and baseline_result.file_name.is_unique
assert baseline_result.resolution.eq(512).all()
assert baseline_result.model_name.eq("korean_PP-OCRv5_mobile_rec").all()
assert baseline_result.det_model_name.eq("PP-OCRv5_mobile_det").all()
labels = pd.read_csv(PROJECT_ROOT / "labels/labels_300.csv", encoding="utf-8-sig", dtype=str).fillna("")
run_labels = labels.set_index("file_name").loc[baseline_result.file_name].reset_index()
assert run_labels.image_id.tolist() == baseline_result.image_id.tolist()
assert run_labels.final_date.tolist() == baseline_result.final_date.tolist()
image_by_name = {p.name: p for p in (PROJECT_ROOT / "data/validation/label_images").rglob("*") if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
run_labels["image_path"] = run_labels.file_name.map(image_by_name)
assert len(run_labels) == 10 and run_labels.image_path.notna().all()
CPU_THREADS = 4
BATCH_SIZE = 12
MAX_LONG_SIDE = 1600
os.environ["PADDLE_PDX_CACHE_HOME"] = str(PROJECT_ROOT / "weights/paddlex")
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "0"
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
print(f"Same baseline images: {len(run_labels)}, CPU threads: {CPU_THREADS}, batch: 12 (saved baseline: batch 6)")


def load_common_input(path: Path, max_long_side: int = MAX_LONG_SIDE) -> np.ndarray:
    """두 엔진에 동일하게 전달할 EXIF 보정 RGB uint8 배열을 만듭니다."""
    with Image.open(path) as src:
        image = ImageOps.exif_transpose(src).convert("RGB")
    long_side = max(image.size)
    if long_side > max_long_side:
        scale = max_long_side / long_side
        new_size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        image = image.resize(new_size, Image.Resampling.LANCZOS)
    return np.asarray(image)

def normalize_digits(value) -> str:
    return re.sub(r"[^0-9]", "", str(value))

def expected_date_variants(row) -> set[str]:
    """완전한 Y/M/D 라벨을 흔한 YMD, DMY, MDY 숫자열 후보로 변환합니다."""
    if str(row["final_date"]).strip().upper() in {"", "NONE", "NAN"}:
        return set()
    try:
        year, month, day = int(row["year"]), int(row["month"]), int(row["day"])
    except (TypeError, ValueError):
        return set()
    y4, y2 = f"{year:04d}", f"{year % 100:02d}"
    mm, dd = f"{month:02d}", f"{day:02d}"
    m, d = str(month), str(day)
    return {
        y4 + mm + dd, y2 + mm + dd, y4 + m + d, y2 + m + d,
        dd + mm + y4, dd + mm + y2, d + m + y4, d + m + y2,
        mm + dd + y4, mm + dd + y2, m + d + y4, m + d + y2,
    }

def date_recalled(text: str, row):
    variants = expected_date_variants(row)
    if not variants:
        return pd.NA
    digits = normalize_digits(text)
    return any(candidate in digits for candidate in variants)

def _paddle_payload(item):
    payload = getattr(item, "json", item)
    if callable(payload):
        payload = payload()
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    return payload if isinstance(payload, dict) else {}

def infer_paddleocr(engine, image_array):
    predictions = list(engine.predict(image_array))
    detections = []
    for item in predictions:
        payload = _paddle_payload(item)
        texts = payload.get("rec_texts", []) or []
        scores = payload.get("rec_scores", []) or []
        boxes = payload.get("rec_polys", payload.get("dt_polys", [])) or []
        for index, text in enumerate(texts):
            confidence = float(scores[index]) if index < len(scores) else None
            box = np.asarray(boxes[index]).tolist() if index < len(boxes) else None
            detections.append({"text": str(text), "confidence": confidence, "box": box})
    return detections

# Batch=12 trial: identical 512px mobile baseline settings except recognition batch size.
MOBILE_REC_MODEL = "korean_PP-OCRv5_mobile_rec"
MOBILE_TRIAL_RESOLUTION = 512
MOBILE_OUTPUT_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch12"

from paddleocr import PaddleOCR

started = time.perf_counter()
mobile_engine = PaddleOCR(
    lang="korean",
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name=MOBILE_REC_MODEL,
    text_recognition_batch_size=BATCH_SIZE,
    text_det_limit_side_len=MOBILE_TRIAL_RESOLUTION,
    text_det_limit_type="max",
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=True,
    cpu_threads=CPU_THREADS,
)
mobile_init_seconds = time.perf_counter() - started
print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] 초기화 완료: {mobile_init_seconds:.2f}초")

mobile_rows = []
for position, (_, label_row) in enumerate(run_labels.iterrows(), start=1):
    image_array = load_common_input(label_row["image_path"], max_long_side=MOBILE_TRIAL_RESOLUTION)
    started = time.perf_counter()
    detections = infer_paddleocr(mobile_engine, image_array)
    inference_seconds = time.perf_counter() - started
    detected_text = " | ".join(item["text"] for item in detections)
    confidences = [item["confidence"] for item in detections if item["confidence"] is not None]
    mobile_rows.append({
        "engine": "paddleocr",
        "model_name": MOBILE_REC_MODEL,
        "det_model_name": "PP-OCRv5_mobile_det",
        "resolution": MOBILE_TRIAL_RESOLUTION,
        "text_recognition_batch_size": BATCH_SIZE,
        "cpu_threads": CPU_THREADS,
        "file_name": label_row["file_name"],
        "image_id": label_row["image_id"],
        "final_date": label_row["final_date"],
        "detected_text": detected_text,
        "date_recalled": date_recalled(detected_text, label_row),
        "mean_confidence": float(np.mean(confidences)) if confidences else np.nan,
        "detection_count": len(detections),
        "inference_seconds": inference_seconds,
        "model_init_seconds": mobile_init_seconds,
        "input_height": image_array.shape[0],
        "input_width": image_array.shape[1],
        "raw_detections_json": json.dumps(detections, ensure_ascii=False),
    })
    print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] {position:>2}/{len(run_labels)} {label_row['file_name']} {inference_seconds:.2f}s")

del mobile_engine
gc.collect()
mobile_result = pd.DataFrame(mobile_rows)
mobile_result.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_results.csv",
    index=False,
    encoding="utf-8-sig",
    mode="x",
)

scored = mobile_result["date_recalled"].notna()
mobile_summary = {
    "engine": "paddleocr",
    "model_name": MOBILE_REC_MODEL,
    "det_model_name": "PP-OCRv5_mobile_det",
    "resolution": MOBILE_TRIAL_RESOLUTION,
        "text_recognition_batch_size": BATCH_SIZE,
        "cpu_threads": CPU_THREADS,
    "images": len(mobile_result),
    "date_token_recall": float(mobile_result.loc[scored, "date_recalled"].astype(bool).mean()),
    "mean_seconds_per_image": float(mobile_result["inference_seconds"].mean()),
    "median_seconds_per_image": float(mobile_result["inference_seconds"].median()),
    "estimated_3352_minutes": float(mobile_result["inference_seconds"].mean() * 3352 / 60),
    "model_init_seconds": mobile_init_seconds,
    "target_seconds_per_image": 0.7,
    "target_met": bool(mobile_result["inference_seconds"].mean() <= 0.7),
}
mean_improvement_fraction = 1 - mobile_summary["mean_seconds_per_image"] / baseline_summary["mean_seconds_per_image"]
recall_preserved = mobile_summary["date_token_recall"] >= baseline_summary["date_token_recall"]
accept_batch12 = bool(mean_improvement_fraction >= 0.10 and recall_preserved)
mobile_summary.update({
    "baseline_batch_size": 6,
    "baseline_results_csv": str(BASELINE_PREFIX.name) + "_results.csv",
    "mean_improvement_percent": 100 * mean_improvement_fraction,
    "required_mean_improvement_percent": 10,
    "recall_preserved": bool(recall_preserved),
    "accept_batch12": accept_batch12,
    "stop_additional_batch_experiments": True,
    "decision": "accept_batch12_final_batch_trial" if accept_batch12 else "retain_batch6_stop_batch_trials",
})

mobile_summary_df = pd.DataFrame([mobile_summary])
mobile_summary_df.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_summary.csv",
    index=False,
    encoding="utf-8-sig",
    mode="x",
)


assert mobile_result.file_name.tolist() == baseline_result.file_name.tolist()
metrics = ["date_token_recall", "mean_seconds_per_image", "median_seconds_per_image", "estimated_3352_minutes"]
batch_comparison = pd.DataFrame([
    {"metric": metric, "baseline_batch6": baseline_summary[metric], "trial_batch12": mobile_summary[metric],
     "batch12_minus_batch6": mobile_summary[metric] - baseline_summary[metric]}
    for metric in metrics
])
batch_comparison.to_csv(f"{TRIAL_PREFIX}_vs_batch6.csv", index=False, encoding="utf-8-sig", mode="x")
print(batch_comparison.to_string(index=False))
print("Mean time reduction (%):", 100 * (1 - mobile_summary["mean_seconds_per_image"] / baseline_summary["mean_seconds_per_image"]))
print("Recall changes:")
print(baseline_result[["file_name", "date_recalled"]].merge(mobile_result[["file_name", "date_recalled"]], on="file_name", suffixes=("_batch6", "_batch12")).to_string(index=False))

print("Final batch experiment decision:")
print("Mean improvement (%):", mobile_summary["mean_improvement_percent"])
print("Recall preserved:", recall_preserved)
if accept_batch12:
    print("Batch12 meets the >=10% mean-time improvement and recall criteria. This is the final batch experiment.")
else:
    print("Batch12 fails the >=10% mean-time improvement and/or recall criteria. Retain 512px batch6 and STOP additional batch experiments.")


Same baseline images: 10, CPU threads: 4, batch: 12 (saved baseline: batch 6)


[paddleocr 512px/korean_PP-OCRv5_mobile_rec] 초기화 완료: 7.64초


metric  baseline_batch6  trial_batch12  batch12_minus_batch6
       date_token_recall         1.000000       1.000000              0.000000
  mean_seconds_per_image         1.729757       2.244666              0.514909
median_seconds_per_image         1.280868       1.578405              0.297537
  estimated_3352_minutes        96.635766     125.402016             28.766251
Mean time reduction (%): -29.767706002893355
Recall changes:
 file_name  date_recalled_batch6  date_recalled_batch12
000007.jpg                  True                   True
000018.jpg                  True                   True
000021.jpg                  True                   True
000026.jpg                  True                   True
000030.jpg                  True                   True
000058.jpg                  True                   True
000060.jpg                  True                   True
000075.jpg                  True                   True
000099.jpg                  True                   True
00

In [1]:
# Detection threshold experiment: baseline 0.6 confirmed by user from omitted argument and OCR.yaml default.
# 512px recognition batch-size experiment: execute this new cell only.
# Saved default-batch baseline (cell 22) is interpreted as actual batch size 6.
# Keep both mobile models, CPU threads=4, preprocessing, scoring, and timing unchanged.
# Never rerun completed experiments; use only the same 10 filenames and order from CSV.
from pathlib import Path
import gc
import json
import os
import re
import time
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (cwd, *cwd.parents) if (p / "predict.ipynb").is_file() and (p / "data").is_dir())
OUTPUT_DIR = PROJECT_ROOT / "data/validation/ocr_comparison"
BASELINE_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec"
TRIAL_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch6_boxthresh07"
assert not list(OUTPUT_DIR.glob(TRIAL_PREFIX.name + "*.csv")), "boxthresh07 result exists: do not rerun"
baseline_result = pd.read_csv(f"{BASELINE_PREFIX}_results.csv", dtype={"image_id": str})
baseline_summary = pd.read_csv(f"{BASELINE_PREFIX}_summary.csv").iloc[0]
assert len(baseline_result) == 10 and baseline_result.file_name.is_unique
assert baseline_result.resolution.eq(512).all()
assert baseline_result.model_name.eq("korean_PP-OCRv5_mobile_rec").all()
assert baseline_result.det_model_name.eq("PP-OCRv5_mobile_det").all()
labels = pd.read_csv(PROJECT_ROOT / "labels/labels_300.csv", encoding="utf-8-sig", dtype=str).fillna("")
run_labels = labels.set_index("file_name").loc[baseline_result.file_name].reset_index()
assert run_labels.image_id.tolist() == baseline_result.image_id.tolist()
assert run_labels.final_date.tolist() == baseline_result.final_date.tolist()
image_by_name = {p.name: p for p in (PROJECT_ROOT / "data/validation/label_images").rglob("*") if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
run_labels["image_path"] = run_labels.file_name.map(image_by_name)
assert len(run_labels) == 10 and run_labels.image_path.notna().all()
CPU_THREADS = 4
BATCH_SIZE = 6
TEXT_DET_BOX_THRESH = 0.7
BASELINE_TEXT_DET_BOX_THRESH = 0.6
MAX_LONG_SIDE = 1600
os.environ["PADDLE_PDX_CACHE_HOME"] = str(PROJECT_ROOT / "weights/paddlex")
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "0"
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
print(f"Same baseline images: {len(run_labels)}, CPU threads: {CPU_THREADS}, batch: 6, predict box threshold: 0.7 (baseline: 0.6)")


def load_common_input(path: Path, max_long_side: int = MAX_LONG_SIDE) -> np.ndarray:
    """두 엔진에 동일하게 전달할 EXIF 보정 RGB uint8 배열을 만듭니다."""
    with Image.open(path) as src:
        image = ImageOps.exif_transpose(src).convert("RGB")
    long_side = max(image.size)
    if long_side > max_long_side:
        scale = max_long_side / long_side
        new_size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        image = image.resize(new_size, Image.Resampling.LANCZOS)
    return np.asarray(image)

def normalize_digits(value) -> str:
    return re.sub(r"[^0-9]", "", str(value))

def expected_date_variants(row) -> set[str]:
    """완전한 Y/M/D 라벨을 흔한 YMD, DMY, MDY 숫자열 후보로 변환합니다."""
    if str(row["final_date"]).strip().upper() in {"", "NONE", "NAN"}:
        return set()
    try:
        year, month, day = int(row["year"]), int(row["month"]), int(row["day"])
    except (TypeError, ValueError):
        return set()
    y4, y2 = f"{year:04d}", f"{year % 100:02d}"
    mm, dd = f"{month:02d}", f"{day:02d}"
    m, d = str(month), str(day)
    return {
        y4 + mm + dd, y2 + mm + dd, y4 + m + d, y2 + m + d,
        dd + mm + y4, dd + mm + y2, d + m + y4, d + m + y2,
        mm + dd + y4, mm + dd + y2, m + d + y4, m + d + y2,
    }

def date_recalled(text: str, row):
    variants = expected_date_variants(row)
    if not variants:
        return pd.NA
    digits = normalize_digits(text)
    return any(candidate in digits for candidate in variants)

def _paddle_payload(item):
    payload = getattr(item, "json", item)
    if callable(payload):
        payload = payload()
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    return payload if isinstance(payload, dict) else {}

def infer_paddleocr(engine, image_array):
    predictions = list(engine.predict(image_array, text_det_box_thresh=TEXT_DET_BOX_THRESH))
    detections = []
    for item in predictions:
        payload = _paddle_payload(item)
        texts = payload.get("rec_texts", []) or []
        scores = payload.get("rec_scores", []) or []
        boxes = payload.get("rec_polys", payload.get("dt_polys", [])) or []
        for index, text in enumerate(texts):
            confidence = float(scores[index]) if index < len(scores) else None
            box = np.asarray(boxes[index]).tolist() if index < len(boxes) else None
            detections.append({"text": str(text), "confidence": confidence, "box": box})
    return detections

# Threshold trial: only override detection box threshold at predict time.
MOBILE_REC_MODEL = "korean_PP-OCRv5_mobile_rec"
MOBILE_TRIAL_RESOLUTION = 512
MOBILE_OUTPUT_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch6_boxthresh07"

from paddleocr import PaddleOCR

started = time.perf_counter()
mobile_engine = PaddleOCR(
    lang="korean",
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name=MOBILE_REC_MODEL,
    text_recognition_batch_size=BATCH_SIZE,
    text_det_limit_side_len=MOBILE_TRIAL_RESOLUTION,
    text_det_limit_type="max",
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=True,
    cpu_threads=CPU_THREADS,
)
mobile_init_seconds = time.perf_counter() - started
print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] 초기화 완료: {mobile_init_seconds:.2f}초")

mobile_rows = []
for position, (_, label_row) in enumerate(run_labels.iterrows(), start=1):
    image_array = load_common_input(label_row["image_path"], max_long_side=MOBILE_TRIAL_RESOLUTION)
    started = time.perf_counter()
    detections = infer_paddleocr(mobile_engine, image_array)
    inference_seconds = time.perf_counter() - started
    detected_text = " | ".join(item["text"] for item in detections)
    confidences = [item["confidence"] for item in detections if item["confidence"] is not None]
    mobile_rows.append({
        "engine": "paddleocr",
        "model_name": MOBILE_REC_MODEL,
        "det_model_name": "PP-OCRv5_mobile_det",
        "resolution": MOBILE_TRIAL_RESOLUTION,
        "text_recognition_batch_size": BATCH_SIZE,
        "cpu_threads": CPU_THREADS,
        "text_det_box_thresh": TEXT_DET_BOX_THRESH,
        "file_name": label_row["file_name"],
        "image_id": label_row["image_id"],
        "final_date": label_row["final_date"],
        "detected_text": detected_text,
        "date_recalled": date_recalled(detected_text, label_row),
        "mean_confidence": float(np.mean(confidences)) if confidences else np.nan,
        "detection_count": len(detections),
        "inference_seconds": inference_seconds,
        "model_init_seconds": mobile_init_seconds,
        "input_height": image_array.shape[0],
        "input_width": image_array.shape[1],
        "raw_detections_json": json.dumps(detections, ensure_ascii=False),
    })
    print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] {position:>2}/{len(run_labels)} {label_row['file_name']} {inference_seconds:.2f}s")

del mobile_engine
gc.collect()
mobile_result = pd.DataFrame(mobile_rows)
mobile_result.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_results.csv",
    index=False,
    encoding="utf-8-sig",
    mode="x",
)

scored = mobile_result["date_recalled"].notna()
mobile_summary = {
    "engine": "paddleocr",
    "model_name": MOBILE_REC_MODEL,
    "det_model_name": "PP-OCRv5_mobile_det",
    "resolution": MOBILE_TRIAL_RESOLUTION,
        "text_recognition_batch_size": BATCH_SIZE,
        "cpu_threads": CPU_THREADS,
        "text_det_box_thresh": TEXT_DET_BOX_THRESH,
    "images": len(mobile_result),
    "date_token_recall": float(mobile_result.loc[scored, "date_recalled"].astype(bool).mean()),
    "mean_seconds_per_image": float(mobile_result["inference_seconds"].mean()),
    "median_seconds_per_image": float(mobile_result["inference_seconds"].median()),
    "estimated_3352_minutes": float(mobile_result["inference_seconds"].mean() * 3352 / 60),
    "model_init_seconds": mobile_init_seconds,
    "target_seconds_per_image": 0.7,
    "target_met": bool(mobile_result["inference_seconds"].mean() <= 0.7),
}
mobile_summary["total_detection_boxes"] = int(mobile_result["detection_count"].sum())
mobile_summary["mean_detection_boxes_per_image"] = float(mobile_result["detection_count"].mean())
baseline_summary["total_detection_boxes"] = int(baseline_result["detection_count"].sum())
baseline_summary["mean_detection_boxes_per_image"] = float(baseline_result["detection_count"].mean())
mobile_summary["baseline_text_det_box_thresh"] = BASELINE_TEXT_DET_BOX_THRESH
mobile_summary["mean_time_reduction_percent"] = 100 * (1 - mobile_summary["mean_seconds_per_image"] / baseline_summary["mean_seconds_per_image"])
mobile_summary["box_count_reduction_percent"] = 100 * (1 - mobile_summary["total_detection_boxes"] / baseline_summary["total_detection_boxes"])
mobile_summary["recall_preserved_at_100_percent"] = bool(mobile_summary["date_token_recall"] == 1.0)
mobile_summary_df = pd.DataFrame([mobile_summary])
mobile_summary_df.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_summary.csv",
    index=False,
    encoding="utf-8-sig",
    mode="x",
)


assert mobile_result.file_name.tolist() == baseline_result.file_name.tolist()
metrics = ["date_token_recall", "mean_seconds_per_image", "median_seconds_per_image", "estimated_3352_minutes", "total_detection_boxes", "mean_detection_boxes_per_image"]
threshold_comparison = pd.DataFrame([
    {"metric": metric, "baseline_thresh06": baseline_summary[metric], "trial_thresh07": mobile_summary[metric],
     "thresh07_minus_thresh06": mobile_summary[metric] - baseline_summary[metric]}
    for metric in metrics
])
threshold_comparison.to_csv(f"{TRIAL_PREFIX}_vs_thresh06.csv", index=False, encoding="utf-8-sig", mode="x")
print(threshold_comparison.to_string(index=False))
print("Mean time reduction (%):", 100 * (1 - mobile_summary["mean_seconds_per_image"] / baseline_summary["mean_seconds_per_image"]))
print("Recall changes:")
print(baseline_result[["file_name", "date_recalled"]].merge(mobile_result[["file_name", "date_recalled"]], on="file_name", suffixes=("_thresh06", "_thresh07")).to_string(index=False))

per_image_comparison = baseline_result[["file_name", "date_recalled", "inference_seconds", "detection_count"]].merge(
    mobile_result[["file_name", "date_recalled", "inference_seconds", "detection_count"]],
    on="file_name", suffixes=("_thresh06", "_thresh07"), validate="one_to_one")
per_image_comparison["baseline_text_det_box_thresh"] = 0.6
per_image_comparison["trial_text_det_box_thresh"] = 0.7
per_image_comparison.to_csv(f"{TRIAL_PREFIX}_per_image_vs_thresh06.csv", index=False, encoding="utf-8-sig", mode="x")
print(per_image_comparison.to_string(index=False))
print("Mean time reduction (%):", mobile_summary["mean_time_reduction_percent"])
print("Box count reduction (%):", mobile_summary["box_count_reduction_percent"])
print("Recall preserved at 100%:", mobile_summary["recall_preserved_at_100_percent"])
print("Single run on 10 images; timing differences do not establish statistical significance.")


Same baseline images: 10, CPU threads: 4, batch: 6, predict box threshold: 0.7 (baseline: 0.6)


[paddleocr 512px/korean_PP-OCRv5_mobile_rec] 초기화 완료: 7.65초


metric  baseline_thresh06  trial_thresh07  thresh07_minus_thresh06
             date_token_recall           1.000000        1.000000                 0.000000
        mean_seconds_per_image           1.729757        1.340528                -0.389229
      median_seconds_per_image           1.280868        1.019209                -0.261659
        estimated_3352_minutes          96.635766       74.890841               -21.744925
         total_detection_boxes         250.000000      155.000000               -95.000000
mean_detection_boxes_per_image          25.000000       15.500000                -9.500000
Mean time reduction (%): 22.50194267155835
Recall changes:
 file_name  date_recalled_thresh06  date_recalled_thresh07
000007.jpg                    True                    True
000018.jpg                    True                    True
000021.jpg                    True                    True
000026.jpg                    True                    True
000030.jpg                    Tru

In [1]:
# Detection threshold experiment: compare predict threshold 0.8 against saved provisional 0.7 baseline.
# 512px recognition batch-size experiment: execute this new cell only.
# Saved default-batch baseline (cell 22) is interpreted as actual batch size 6.
# Keep both mobile models, CPU threads=4, preprocessing, scoring, and timing unchanged.
# Never rerun completed experiments; use only the same 10 filenames and order from CSV.
from pathlib import Path
import gc
import json
import os
import re
import time
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (cwd, *cwd.parents) if (p / "predict.ipynb").is_file() and (p / "data").is_dir())
OUTPUT_DIR = PROJECT_ROOT / "data/validation/ocr_comparison"
BASELINE_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch6_boxthresh07"
TRIAL_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch6_boxthresh08"
assert not list(OUTPUT_DIR.glob(TRIAL_PREFIX.name + "*.csv")), "boxthresh08 result exists: do not rerun"
baseline_result = pd.read_csv(f"{BASELINE_PREFIX}_results.csv", dtype={"image_id": str})
baseline_summary = pd.read_csv(f"{BASELINE_PREFIX}_summary.csv").iloc[0]
assert len(baseline_result) == 10 and baseline_result.file_name.is_unique
assert baseline_result.resolution.eq(512).all()
assert baseline_result.text_det_box_thresh.eq(0.7).all()
assert baseline_result.text_recognition_batch_size.eq(6).all()
assert baseline_result.cpu_threads.eq(4).all()
assert baseline_result.model_name.eq("korean_PP-OCRv5_mobile_rec").all()
assert baseline_result.det_model_name.eq("PP-OCRv5_mobile_det").all()
labels = pd.read_csv(PROJECT_ROOT / "labels/labels_300.csv", encoding="utf-8-sig", dtype=str).fillna("")
run_labels = labels.set_index("file_name").loc[baseline_result.file_name].reset_index()
assert run_labels.image_id.tolist() == baseline_result.image_id.tolist()
assert run_labels.final_date.tolist() == baseline_result.final_date.tolist()
image_by_name = {p.name: p for p in (PROJECT_ROOT / "data/validation/label_images").rglob("*") if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
run_labels["image_path"] = run_labels.file_name.map(image_by_name)
assert len(run_labels) == 10 and run_labels.image_path.notna().all()
CPU_THREADS = 4
BATCH_SIZE = 6
TEXT_DET_BOX_THRESH = 0.8
BASELINE_TEXT_DET_BOX_THRESH = 0.7
MAX_LONG_SIDE = 1600
os.environ["PADDLE_PDX_CACHE_HOME"] = str(PROJECT_ROOT / "weights/paddlex")
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "0"
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
print(f"Same baseline images: {len(run_labels)}, CPU threads: {CPU_THREADS}, batch: 6, predict box threshold: 0.8 (baseline: 0.7)")


def load_common_input(path: Path, max_long_side: int = MAX_LONG_SIDE) -> np.ndarray:
    """두 엔진에 동일하게 전달할 EXIF 보정 RGB uint8 배열을 만듭니다."""
    with Image.open(path) as src:
        image = ImageOps.exif_transpose(src).convert("RGB")
    long_side = max(image.size)
    if long_side > max_long_side:
        scale = max_long_side / long_side
        new_size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        image = image.resize(new_size, Image.Resampling.LANCZOS)
    return np.asarray(image)

def normalize_digits(value) -> str:
    return re.sub(r"[^0-9]", "", str(value))

def expected_date_variants(row) -> set[str]:
    """완전한 Y/M/D 라벨을 흔한 YMD, DMY, MDY 숫자열 후보로 변환합니다."""
    if str(row["final_date"]).strip().upper() in {"", "NONE", "NAN"}:
        return set()
    try:
        year, month, day = int(row["year"]), int(row["month"]), int(row["day"])
    except (TypeError, ValueError):
        return set()
    y4, y2 = f"{year:04d}", f"{year % 100:02d}"
    mm, dd = f"{month:02d}", f"{day:02d}"
    m, d = str(month), str(day)
    return {
        y4 + mm + dd, y2 + mm + dd, y4 + m + d, y2 + m + d,
        dd + mm + y4, dd + mm + y2, d + m + y4, d + m + y2,
        mm + dd + y4, mm + dd + y2, m + d + y4, m + d + y2,
    }

def date_recalled(text: str, row):
    variants = expected_date_variants(row)
    if not variants:
        return pd.NA
    digits = normalize_digits(text)
    return any(candidate in digits for candidate in variants)

def _paddle_payload(item):
    payload = getattr(item, "json", item)
    if callable(payload):
        payload = payload()
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    return payload if isinstance(payload, dict) else {}

def infer_paddleocr(engine, image_array):
    predictions = list(engine.predict(image_array, text_det_box_thresh=TEXT_DET_BOX_THRESH))
    detections = []
    for item in predictions:
        payload = _paddle_payload(item)
        texts = payload.get("rec_texts", []) or []
        scores = payload.get("rec_scores", []) or []
        boxes = payload.get("rec_polys", payload.get("dt_polys", [])) or []
        for index, text in enumerate(texts):
            confidence = float(scores[index]) if index < len(scores) else None
            box = np.asarray(boxes[index]).tolist() if index < len(boxes) else None
            detections.append({"text": str(text), "confidence": confidence, "box": box})
    return detections

# Threshold trial: only override detection box threshold at predict time.
MOBILE_REC_MODEL = "korean_PP-OCRv5_mobile_rec"
MOBILE_TRIAL_RESOLUTION = 512
MOBILE_OUTPUT_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch6_boxthresh08"

from paddleocr import PaddleOCR

started = time.perf_counter()
mobile_engine = PaddleOCR(
    lang="korean",
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name=MOBILE_REC_MODEL,
    text_recognition_batch_size=BATCH_SIZE,
    text_det_limit_side_len=MOBILE_TRIAL_RESOLUTION,
    text_det_limit_type="max",
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=True,
    cpu_threads=CPU_THREADS,
)
mobile_init_seconds = time.perf_counter() - started
print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] 초기화 완료: {mobile_init_seconds:.2f}초")

mobile_rows = []
for position, (_, label_row) in enumerate(run_labels.iterrows(), start=1):
    image_array = load_common_input(label_row["image_path"], max_long_side=MOBILE_TRIAL_RESOLUTION)
    started = time.perf_counter()
    detections = infer_paddleocr(mobile_engine, image_array)
    inference_seconds = time.perf_counter() - started
    detected_text = " | ".join(item["text"] for item in detections)
    confidences = [item["confidence"] for item in detections if item["confidence"] is not None]
    mobile_rows.append({
        "engine": "paddleocr",
        "model_name": MOBILE_REC_MODEL,
        "det_model_name": "PP-OCRv5_mobile_det",
        "resolution": MOBILE_TRIAL_RESOLUTION,
        "text_recognition_batch_size": BATCH_SIZE,
        "cpu_threads": CPU_THREADS,
        "text_det_box_thresh": TEXT_DET_BOX_THRESH,
        "file_name": label_row["file_name"],
        "image_id": label_row["image_id"],
        "final_date": label_row["final_date"],
        "detected_text": detected_text,
        "date_recalled": date_recalled(detected_text, label_row),
        "mean_confidence": float(np.mean(confidences)) if confidences else np.nan,
        "detection_count": len(detections),
        "inference_seconds": inference_seconds,
        "model_init_seconds": mobile_init_seconds,
        "input_height": image_array.shape[0],
        "input_width": image_array.shape[1],
        "raw_detections_json": json.dumps(detections, ensure_ascii=False),
    })
    print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] {position:>2}/{len(run_labels)} {label_row['file_name']} {inference_seconds:.2f}s")

del mobile_engine
gc.collect()
mobile_result = pd.DataFrame(mobile_rows)
mobile_result.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_results.csv",
    index=False,
    encoding="utf-8-sig",
    mode="x",
)

scored = mobile_result["date_recalled"].notna()
mobile_summary = {
    "engine": "paddleocr",
    "model_name": MOBILE_REC_MODEL,
    "det_model_name": "PP-OCRv5_mobile_det",
    "resolution": MOBILE_TRIAL_RESOLUTION,
        "text_recognition_batch_size": BATCH_SIZE,
        "cpu_threads": CPU_THREADS,
        "text_det_box_thresh": TEXT_DET_BOX_THRESH,
    "images": len(mobile_result),
    "date_token_recall": float(mobile_result.loc[scored, "date_recalled"].astype(bool).mean()),
    "mean_seconds_per_image": float(mobile_result["inference_seconds"].mean()),
    "median_seconds_per_image": float(mobile_result["inference_seconds"].median()),
    "estimated_3352_minutes": float(mobile_result["inference_seconds"].mean() * 3352 / 60),
    "model_init_seconds": mobile_init_seconds,
    "target_seconds_per_image": 0.7,
    "target_met": bool(mobile_result["inference_seconds"].mean() <= 0.7),
}
mobile_summary["total_detection_boxes"] = int(mobile_result["detection_count"].sum())
mobile_summary["mean_detection_boxes_per_image"] = float(mobile_result["detection_count"].mean())
baseline_summary["total_detection_boxes"] = int(baseline_result["detection_count"].sum())
baseline_summary["mean_detection_boxes_per_image"] = float(baseline_result["detection_count"].mean())
mobile_summary["baseline_text_det_box_thresh"] = BASELINE_TEXT_DET_BOX_THRESH
mobile_summary["mean_time_reduction_percent"] = 100 * (1 - mobile_summary["mean_seconds_per_image"] / baseline_summary["mean_seconds_per_image"])
mobile_summary["box_count_reduction_percent"] = 100 * (1 - mobile_summary["total_detection_boxes"] / baseline_summary["total_detection_boxes"])
mobile_summary["recall_preserved_at_100_percent"] = bool(mobile_summary["date_token_recall"] == 1.0)
failed_images = mobile_result.loc[mobile_result["date_recalled"].eq(False), "file_name"].tolist()
recall_dropped = mobile_summary["date_token_recall"] < baseline_summary["date_token_recall"]
candidate = bool(mobile_summary["date_token_recall"] == 1.0 and mobile_summary["mean_seconds_per_image"] <= 0.9 * baseline_summary["mean_seconds_per_image"])
decision = "reject_08_recall_drop" if recall_dropped else ("candidate_for_expanded_validation" if candidate else "retain_07_insufficient_speed_improvement")
mobile_summary.update({"failed_images_json": json.dumps(failed_images), "decision": decision, "expanded_validation_candidate": candidate})
mobile_summary_df = pd.DataFrame([mobile_summary])
mobile_summary_df.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_summary.csv",
    index=False,
    encoding="utf-8-sig",
    mode="x",
)


assert mobile_result.file_name.tolist() == baseline_result.file_name.tolist()
metrics = ["date_token_recall", "mean_seconds_per_image", "median_seconds_per_image", "estimated_3352_minutes", "total_detection_boxes", "mean_detection_boxes_per_image"]
threshold_comparison = pd.DataFrame([
    {"metric": metric, "baseline_thresh07": baseline_summary[metric], "trial_thresh08": mobile_summary[metric],
     "thresh08_minus_thresh07": mobile_summary[metric] - baseline_summary[metric]}
    for metric in metrics
])
threshold_comparison.to_csv(f"{TRIAL_PREFIX}_vs_thresh07.csv", index=False, encoding="utf-8-sig", mode="x")
print(threshold_comparison.to_string(index=False))
print("Mean time reduction (%):", 100 * (1 - mobile_summary["mean_seconds_per_image"] / baseline_summary["mean_seconds_per_image"]))
print("Recall changes:")
print(baseline_result[["file_name", "date_recalled"]].merge(mobile_result[["file_name", "date_recalled"]], on="file_name", suffixes=("_thresh07", "_thresh08")).to_string(index=False))

per_image_comparison = baseline_result[["file_name", "date_recalled", "inference_seconds", "detection_count"]].merge(
    mobile_result[["file_name", "date_recalled", "inference_seconds", "detection_count"]],
    on="file_name", suffixes=("_thresh07", "_thresh08"), validate="one_to_one")
per_image_comparison["baseline_text_det_box_thresh"] = 0.7
per_image_comparison["trial_text_det_box_thresh"] = 0.8
per_image_comparison.to_csv(f"{TRIAL_PREFIX}_per_image_vs_thresh07.csv", index=False, encoding="utf-8-sig", mode="x")
print(per_image_comparison.to_string(index=False))
print("Mean time reduction (%):", mobile_summary["mean_time_reduction_percent"])
print("Box count reduction (%):", mobile_summary["box_count_reduction_percent"])
print("Recall preserved at 100%:", mobile_summary["recall_preserved_at_100_percent"])
print("Single run on 10 images; timing differences do not establish statistical significance.")

failed_result = mobile_result.loc[mobile_result["date_recalled"].eq(False), ["file_name", "final_date", "detected_text", "date_recalled", "text_det_box_thresh"]]
failed_result.to_csv(f"{TRIAL_PREFIX}_failures.csv", index=False, encoding="utf-8-sig", mode="x")
print("Failed images:", failed_images)
print("Decision:", decision)


Same baseline images: 10, CPU threads: 4, batch: 6, predict box threshold: 0.8 (baseline: 0.7)


[paddleocr 512px/korean_PP-OCRv5_mobile_rec] 초기화 완료: 8.23초


metric  baseline_thresh07  trial_thresh08  thresh08_minus_thresh07
             date_token_recall           1.000000        0.600000                -0.400000
        mean_seconds_per_image           1.340528        0.863856                -0.476672
      median_seconds_per_image           1.019209        0.751883                -0.267326
        estimated_3352_minutes          74.890841       48.260778               -26.630063
         total_detection_boxes         155.000000       57.000000               -98.000000
mean_detection_boxes_per_image          15.500000        5.700000                -9.800000
Mean time reduction (%): 35.558504260636184
Recall changes:
 file_name  date_recalled_thresh07  date_recalled_thresh08
000007.jpg                    True                    True
000018.jpg                    True                    True
000021.jpg                    True                    True
000026.jpg                    True                    True
000030.jpg                    Tr

In [1]:
# Detection threshold experiment: compare predict threshold 0.75 against saved provisional 0.7 baseline.
# 512px recognition batch-size experiment: execute this new cell only.
# Saved default-batch baseline (cell 22) is interpreted as actual batch size 6.
# Keep both mobile models, CPU threads=4, preprocessing, scoring, and timing unchanged.
# Never rerun completed experiments; use only the same 10 filenames and order from CSV.
from pathlib import Path
import gc
import json
import os
import re
import time
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (cwd, *cwd.parents) if (p / "predict.ipynb").is_file() and (p / "data").is_dir())
OUTPUT_DIR = PROJECT_ROOT / "data/validation/ocr_comparison"
BASELINE_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch6_boxthresh07"
TRIAL_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch6_boxthresh075"
assert not list(OUTPUT_DIR.glob(TRIAL_PREFIX.name + "*.csv")), "boxthresh075 result exists: do not rerun"
baseline_result = pd.read_csv(f"{BASELINE_PREFIX}_results.csv", dtype={"image_id": str})
baseline_summary = pd.read_csv(f"{BASELINE_PREFIX}_summary.csv").iloc[0]
assert len(baseline_result) == 10 and baseline_result.file_name.is_unique
assert baseline_result.resolution.eq(512).all()
assert baseline_result.text_det_box_thresh.eq(0.7).all()
assert baseline_result.text_recognition_batch_size.eq(6).all()
assert baseline_result.cpu_threads.eq(4).all()
assert baseline_result.model_name.eq("korean_PP-OCRv5_mobile_rec").all()
assert baseline_result.det_model_name.eq("PP-OCRv5_mobile_det").all()
labels = pd.read_csv(PROJECT_ROOT / "labels/labels_300.csv", encoding="utf-8-sig", dtype=str).fillna("")
run_labels = labels.set_index("file_name").loc[baseline_result.file_name].reset_index()
assert run_labels.image_id.tolist() == baseline_result.image_id.tolist()
assert run_labels.final_date.tolist() == baseline_result.final_date.tolist()
image_by_name = {p.name: p for p in (PROJECT_ROOT / "data/validation/label_images").rglob("*") if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
run_labels["image_path"] = run_labels.file_name.map(image_by_name)
assert len(run_labels) == 10 and run_labels.image_path.notna().all()
CPU_THREADS = 4
BATCH_SIZE = 6
TEXT_DET_BOX_THRESH = 0.75
BASELINE_TEXT_DET_BOX_THRESH = 0.7
MAX_LONG_SIDE = 1600
os.environ["PADDLE_PDX_CACHE_HOME"] = str(PROJECT_ROOT / "weights/paddlex")
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "0"
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
print(f"Same baseline images: {len(run_labels)}, CPU threads: {CPU_THREADS}, batch: 6, predict box threshold: 0.75 (baseline: 0.7)")


def load_common_input(path: Path, max_long_side: int = MAX_LONG_SIDE) -> np.ndarray:
    """두 엔진에 동일하게 전달할 EXIF 보정 RGB uint8 배열을 만듭니다."""
    with Image.open(path) as src:
        image = ImageOps.exif_transpose(src).convert("RGB")
    long_side = max(image.size)
    if long_side > max_long_side:
        scale = max_long_side / long_side
        new_size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        image = image.resize(new_size, Image.Resampling.LANCZOS)
    return np.asarray(image)

def normalize_digits(value) -> str:
    return re.sub(r"[^0-9]", "", str(value))

def expected_date_variants(row) -> set[str]:
    """완전한 Y/M/D 라벨을 흔한 YMD, DMY, MDY 숫자열 후보로 변환합니다."""
    if str(row["final_date"]).strip().upper() in {"", "NONE", "NAN"}:
        return set()
    try:
        year, month, day = int(row["year"]), int(row["month"]), int(row["day"])
    except (TypeError, ValueError):
        return set()
    y4, y2 = f"{year:04d}", f"{year % 100:02d}"
    mm, dd = f"{month:02d}", f"{day:02d}"
    m, d = str(month), str(day)
    return {
        y4 + mm + dd, y2 + mm + dd, y4 + m + d, y2 + m + d,
        dd + mm + y4, dd + mm + y2, d + m + y4, d + m + y2,
        mm + dd + y4, mm + dd + y2, m + d + y4, m + d + y2,
    }

def date_recalled(text: str, row):
    variants = expected_date_variants(row)
    if not variants:
        return pd.NA
    digits = normalize_digits(text)
    return any(candidate in digits for candidate in variants)

def _paddle_payload(item):
    payload = getattr(item, "json", item)
    if callable(payload):
        payload = payload()
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    return payload if isinstance(payload, dict) else {}

def infer_paddleocr(engine, image_array):
    predictions = list(engine.predict(image_array, text_det_box_thresh=TEXT_DET_BOX_THRESH))
    detections = []
    for item in predictions:
        payload = _paddle_payload(item)
        texts = payload.get("rec_texts", []) or []
        scores = payload.get("rec_scores", []) or []
        boxes = payload.get("rec_polys", payload.get("dt_polys", [])) or []
        for index, text in enumerate(texts):
            confidence = float(scores[index]) if index < len(scores) else None
            box = np.asarray(boxes[index]).tolist() if index < len(boxes) else None
            detections.append({"text": str(text), "confidence": confidence, "box": box})
    return detections

# Threshold trial: only override detection box threshold at predict time.
MOBILE_REC_MODEL = "korean_PP-OCRv5_mobile_rec"
MOBILE_TRIAL_RESOLUTION = 512
MOBILE_OUTPUT_PREFIX = OUTPUT_DIR / "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch6_boxthresh075"

from paddleocr import PaddleOCR

started = time.perf_counter()
mobile_engine = PaddleOCR(
    lang="korean",
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name=MOBILE_REC_MODEL,
    text_recognition_batch_size=BATCH_SIZE,
    text_det_limit_side_len=MOBILE_TRIAL_RESOLUTION,
    text_det_limit_type="max",
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=True,
    cpu_threads=CPU_THREADS,
)
mobile_init_seconds = time.perf_counter() - started
print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] 초기화 완료: {mobile_init_seconds:.2f}초")

mobile_rows = []
for position, (_, label_row) in enumerate(run_labels.iterrows(), start=1):
    image_array = load_common_input(label_row["image_path"], max_long_side=MOBILE_TRIAL_RESOLUTION)
    started = time.perf_counter()
    detections = infer_paddleocr(mobile_engine, image_array)
    inference_seconds = time.perf_counter() - started
    detected_text = " | ".join(item["text"] for item in detections)
    confidences = [item["confidence"] for item in detections if item["confidence"] is not None]
    mobile_rows.append({
        "engine": "paddleocr",
        "model_name": MOBILE_REC_MODEL,
        "det_model_name": "PP-OCRv5_mobile_det",
        "resolution": MOBILE_TRIAL_RESOLUTION,
        "text_recognition_batch_size": BATCH_SIZE,
        "cpu_threads": CPU_THREADS,
        "text_det_box_thresh": TEXT_DET_BOX_THRESH,
        "file_name": label_row["file_name"],
        "image_id": label_row["image_id"],
        "final_date": label_row["final_date"],
        "detected_text": detected_text,
        "date_recalled": date_recalled(detected_text, label_row),
        "mean_confidence": float(np.mean(confidences)) if confidences else np.nan,
        "detection_count": len(detections),
        "inference_seconds": inference_seconds,
        "model_init_seconds": mobile_init_seconds,
        "input_height": image_array.shape[0],
        "input_width": image_array.shape[1],
        "raw_detections_json": json.dumps(detections, ensure_ascii=False),
    })
    print(f"[paddleocr {MOBILE_TRIAL_RESOLUTION}px/{MOBILE_REC_MODEL}] {position:>2}/{len(run_labels)} {label_row['file_name']} {inference_seconds:.2f}s")

del mobile_engine
gc.collect()
mobile_result = pd.DataFrame(mobile_rows)
mobile_result.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_results.csv",
    index=False,
    encoding="utf-8-sig",
    mode="x",
)

scored = mobile_result["date_recalled"].notna()
mobile_summary = {
    "engine": "paddleocr",
    "model_name": MOBILE_REC_MODEL,
    "det_model_name": "PP-OCRv5_mobile_det",
    "resolution": MOBILE_TRIAL_RESOLUTION,
        "text_recognition_batch_size": BATCH_SIZE,
        "cpu_threads": CPU_THREADS,
        "text_det_box_thresh": TEXT_DET_BOX_THRESH,
    "images": len(mobile_result),
    "date_token_recall": float(mobile_result.loc[scored, "date_recalled"].astype(bool).mean()),
    "mean_seconds_per_image": float(mobile_result["inference_seconds"].mean()),
    "median_seconds_per_image": float(mobile_result["inference_seconds"].median()),
    "estimated_3352_minutes": float(mobile_result["inference_seconds"].mean() * 3352 / 60),
    "model_init_seconds": mobile_init_seconds,
    "target_seconds_per_image": 0.7,
    "target_met": bool(mobile_result["inference_seconds"].mean() <= 0.7),
}
mobile_summary["total_detection_boxes"] = int(mobile_result["detection_count"].sum())
mobile_summary["mean_detection_boxes_per_image"] = float(mobile_result["detection_count"].mean())
baseline_summary["total_detection_boxes"] = int(baseline_result["detection_count"].sum())
baseline_summary["mean_detection_boxes_per_image"] = float(baseline_result["detection_count"].mean())
mobile_summary["baseline_text_det_box_thresh"] = BASELINE_TEXT_DET_BOX_THRESH
mobile_summary["mean_time_reduction_percent"] = 100 * (1 - mobile_summary["mean_seconds_per_image"] / baseline_summary["mean_seconds_per_image"])
mobile_summary["box_count_reduction_percent"] = 100 * (1 - mobile_summary["total_detection_boxes"] / baseline_summary["total_detection_boxes"])
mobile_summary["recall_preserved_at_100_percent"] = bool(mobile_summary["date_token_recall"] == 1.0)
failed_images = mobile_result.loc[mobile_result["date_recalled"].eq(False), "file_name"].tolist()
recall_dropped = mobile_summary["date_token_recall"] != 1.0
candidate = bool(mobile_summary["date_token_recall"] == 1.0 and mobile_summary["mean_seconds_per_image"] <= 0.9 * baseline_summary["mean_seconds_per_image"])
decision = "reject_075_recall_not_100_percent" if recall_dropped else ("candidate_for_expanded_validation" if candidate else "retain_07_insufficient_speed_improvement")
mobile_summary.update({"failed_images_json": json.dumps(failed_images), "decision": decision, "expanded_validation_candidate": candidate, "stop_additional_threshold_experiments": True})
mobile_summary_df = pd.DataFrame([mobile_summary])
mobile_summary_df.to_csv(
    f"{MOBILE_OUTPUT_PREFIX}_summary.csv",
    index=False,
    encoding="utf-8-sig",
    mode="x",
)


assert mobile_result.file_name.tolist() == baseline_result.file_name.tolist()
metrics = ["date_token_recall", "mean_seconds_per_image", "median_seconds_per_image", "estimated_3352_minutes", "total_detection_boxes", "mean_detection_boxes_per_image"]
threshold_comparison = pd.DataFrame([
    {"metric": metric, "baseline_thresh07": baseline_summary[metric], "trial_thresh075": mobile_summary[metric],
     "thresh075_minus_thresh07": mobile_summary[metric] - baseline_summary[metric]}
    for metric in metrics
])
threshold_comparison.to_csv(f"{TRIAL_PREFIX}_vs_thresh07.csv", index=False, encoding="utf-8-sig", mode="x")
print(threshold_comparison.to_string(index=False))
print("Mean time reduction (%):", 100 * (1 - mobile_summary["mean_seconds_per_image"] / baseline_summary["mean_seconds_per_image"]))
print("Recall changes:")
print(baseline_result[["file_name", "date_recalled"]].merge(mobile_result[["file_name", "date_recalled"]], on="file_name", suffixes=("_thresh07", "_thresh075")).to_string(index=False))

per_image_comparison = baseline_result[["file_name", "date_recalled", "inference_seconds", "detection_count"]].merge(
    mobile_result[["file_name", "date_recalled", "inference_seconds", "detection_count"]],
    on="file_name", suffixes=("_thresh07", "_thresh075"), validate="one_to_one")
per_image_comparison["baseline_text_det_box_thresh"] = 0.7
per_image_comparison["trial_text_det_box_thresh"] = 0.75
per_image_comparison.to_csv(f"{TRIAL_PREFIX}_per_image_vs_thresh07.csv", index=False, encoding="utf-8-sig", mode="x")
print(per_image_comparison.to_string(index=False))
print("Mean time reduction (%):", mobile_summary["mean_time_reduction_percent"])
print("Box count reduction (%):", mobile_summary["box_count_reduction_percent"])
print("Recall preserved at 100%:", mobile_summary["recall_preserved_at_100_percent"])
print("Single run on 10 images; timing differences do not establish statistical significance.")

failed_result = mobile_result.loc[mobile_result["date_recalled"].eq(False), ["file_name", "final_date", "detected_text", "date_recalled", "text_det_box_thresh"]]
failed_result.to_csv(f"{TRIAL_PREFIX}_failures.csv", index=False, encoding="utf-8-sig", mode="x")
print("Failed images:", failed_images)
print("Decision:", decision)

print("Final single-threshold experiment complete. STOP additional threshold experiments.")


Same baseline images: 10, CPU threads: 4, batch: 6, predict box threshold: 0.75 (baseline: 0.7)


[paddleocr 512px/korean_PP-OCRv5_mobile_rec] 초기화 완료: 9.78초


metric  baseline_thresh07  trial_thresh075  thresh075_minus_thresh07
             date_token_recall           1.000000         0.900000                 -0.100000
        mean_seconds_per_image           1.340528         1.240019                 -0.100510
      median_seconds_per_image           1.019209         0.872548                 -0.146661
        estimated_3352_minutes          74.890841        69.275706                 -5.615135
         total_detection_boxes         155.000000        96.000000                -59.000000
mean_detection_boxes_per_image          15.500000         9.600000                 -5.900000
Mean time reduction (%): 7.497758831164536
Recall changes:
 file_name  date_recalled_thresh07  date_recalled_thresh075
000007.jpg                    True                     True
000018.jpg                    True                     True
000021.jpg                    True                     True
000026.jpg                    True                     True
000030.jpg    

In [1]:
# Bounded validation: cached 10-image simulation, seeded 30/50 subset, failure-only preprocessing.
# Fallback uses OCR text/date validity ONLY, never ground-truth labels.
from pathlib import Path
import os, re, json, time, gc, datetime
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance, ImageFilter

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'predict.ipynb').is_file() and (p / 'data').is_dir())
OUT = ROOT / "data/validation/ocr_comparison"
PREFIX = "paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch6_boxthresh"
RUN = "bounded_seed42_validation"
assert not list(OUT.glob(RUN + "*.csv")), "Existing run artifacts: inspect and resume, never overwrite"
DEADLINE = time.monotonic() + 42 * 60
MAX_LONG_SIDE = 512
CPU_THREADS = 4
errors = 0

def save(frame, suffix):
    frame.to_csv(OUT / f"{RUN}_{suffix}.csv", index=False, encoding="utf-8-sig", mode="x")

def candidates(text):
    # Full calendar dates, separated YMD/DMY/MDY and compact YYYYMMDD/YYMMDD.
    # Inspect each OCR text segment independently; do not concatenate unrelated digits.
    found = set()
    def add(y, m, d):
        y, m, d = int(y), int(m), int(d)
        if y < 100:
            y += 2000
        try:
            if 1900 <= y <= 2099:
                found.add(datetime.date(y, m, d).isoformat())
        except ValueError:
            pass
    for segment in str(text).split('|'):
        for y, m, d in re.findall(r'(?<!\d)((?:19|20)\d{2}|\d{2})\s*[./년-]\s*(\d{1,2})\s*[./월-]\s*(\d{1,2})(?!\d)', segment):
            add(y, m, d)
        for a, b, y in re.findall(r'(?<!\d)(\d{1,2})\s*[./-]\s*(\d{1,2})\s*[./-]\s*((?:19|20)\d{2})(?!\d)', segment):
            add(y, b, a)
            add(y, a, b)
        for digits in re.findall(r'(?<!\d)(?:\d{8}|\d{6})(?!\d)', segment):
            add(digits[:-4], digits[-4:-2], digits[-2:])
    return sorted(found)

saved = {}
for tag in ['07', '075', '08']:
    frame = pd.read_csv(OUT / f"{PREFIX}{tag}_results.csv", dtype={"image_id": str})
    frame['date_recalled'] = frame.date_recalled.astype(str).str.lower().eq('true')
    assert len(frame) == 10 and frame.file_name.is_unique
    saved[tag] = frame.set_index('file_name')
base = saved['07']
simulation = []
details = []
for tag in ['07', '075', '08']:
    frame = saved[tag].loc[base.index]
    retry = frame.detected_text.map(lambda text: len(candidates(text)) == 0) if tag != '07' else pd.Series(False, index=frame.index)
    final_recall = frame.date_recalled | (retry & base.date_recalled)
    total = float(frame.inference_seconds.sum() + base.loc[retry, 'inference_seconds'].sum())
    simulation.append(dict(strategy=tag, recall=float(final_recall.mean()), total_seconds=total, mean_seconds=total/10,
                           estimated_3352_minutes=total/10*3352/60, retries=int(retry.sum()),
                           retry_files=json.dumps(frame.index[retry].tolist()),
                           candidate=bool(final_recall.mean() >= base.date_recalled.mean() and total < base.inference_seconds.sum())))
    for name in frame.index:
        details.append(dict(strategy=tag, file_name=name, date_candidates=json.dumps(candidates(frame.loc[name,'detected_text'])),
                            fallback=bool(retry.loc[name]), first_seconds=float(frame.loc[name,'inference_seconds']),
                            fallback_seconds=float(base.loc[name,'inference_seconds']) if retry.loc[name] else 0,
                            final_recalled=bool(final_recall.loc[name])))
sim = pd.DataFrame(simulation)
save(sim, 'adaptive_simulation')
save(pd.DataFrame(details), 'adaptive_per_image')
eligible = sim.loc[sim.candidate]
selected = str(eligible.sort_values('total_seconds').iloc[0].strategy) if len(eligible) else '07'
threshold = {'07':0.7, '075':0.75, '08':0.8}[selected]
print('SIMULATION', sim.to_string(index=False), 'SELECTED', selected, flush=True)

labels = pd.read_csv(ROOT / 'labels/labels_300.csv', dtype=str, encoding='utf-8-sig').fillna('')
labels = labels.sort_values('image_id', key=lambda s: pd.to_numeric(s)).reset_index(drop=True)
assert len(labels) == 300 and labels.file_name.is_unique
remaining = labels.loc[~labels.file_name.isin(base.index)].sample(n=40, random_state=42)
fixed = pd.concat([labels.set_index('file_name').loc[base.index].reset_index(), remaining], ignore_index=True)
fixed['subset_order'] = range(1, 51)
fixed['included_in_30'] = fixed.subset_order.le(30)
save(fixed, 'fixed_subset50')
image_map = {p.name:p for p in (ROOT/'data/validation/label_images').rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png'}}
fixed['image_path'] = fixed.file_name.map(image_map)
assert fixed.image_path.notna().all()
os.environ['PADDLE_PDX_CACHE_HOME'] = str(ROOT/'weights/paddlex')
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = '1'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'

def load_common_input(path: Path, max_long_side: int = MAX_LONG_SIDE) -> np.ndarray:
    """두 엔진에 동일하게 전달할 EXIF 보정 RGB uint8 배열을 만듭니다."""
    with Image.open(path) as src:
        image = ImageOps.exif_transpose(src).convert("RGB")
    long_side = max(image.size)
    if long_side > max_long_side:
        scale = max_long_side / long_side
        new_size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        image = image.resize(new_size, Image.Resampling.LANCZOS)
    return np.asarray(image)

def normalize_digits(value) -> str:
    return re.sub(r"[^0-9]", "", str(value))

def expected_date_variants(row) -> set[str]:
    """완전한 Y/M/D 라벨을 흔한 YMD, DMY, MDY 숫자열 후보로 변환합니다."""
    if str(row["final_date"]).strip().upper() in {"", "NONE", "NAN"}:
        return set()
    try:
        year, month, day = int(row["year"]), int(row["month"]), int(row["day"])
    except (TypeError, ValueError):
        return set()
    y4, y2 = f"{year:04d}", f"{year % 100:02d}"
    mm, dd = f"{month:02d}", f"{day:02d}"
    m, d = str(month), str(day)
    return {
        y4 + mm + dd, y2 + mm + dd, y4 + m + d, y2 + m + d,
        dd + mm + y4, dd + mm + y2, d + m + y4, d + m + y2,
        mm + dd + y4, mm + dd + y2, m + d + y4, m + d + y2,
    }

def date_recalled(text: str, row):
    variants = expected_date_variants(row)
    if not variants:
        return pd.NA
    digits = normalize_digits(text)
    return any(candidate in digits for candidate in variants)

def _paddle_payload(item):
    payload = getattr(item, "json", item)
    if callable(payload):
        payload = payload()
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    return payload if isinstance(payload, dict) else {}

from paddleocr import PaddleOCR
det_dir = ROOT/'weights/paddlex/official_models/PP-OCRv5_mobile_det'
rec_dir = ROOT/'weights/paddlex/official_models/korean_PP-OCRv5_mobile_rec'
assert det_dir.is_dir() and rec_dir.is_dir(), 'Local models required; no download allowed'
engine = PaddleOCR(lang='korean', text_detection_model_name='PP-OCRv5_mobile_det',
    text_recognition_model_name='korean_PP-OCRv5_mobile_rec', text_detection_model_dir=str(det_dir),
    text_recognition_model_dir=str(rec_dir), text_recognition_batch_size=6, text_det_limit_side_len=512,
    text_det_limit_type='max', device='cpu', use_doc_orientation_classify=False,
    use_doc_unwarping=False, use_textline_orientation=False, enable_mkldnn=True, cpu_threads=4)

def infer(array, thresh):
    global errors
    if time.monotonic() >= DEADLINE:
        raise TimeoutError('Time budget reached; safely stopping')
    started = time.perf_counter()
    try:
        predictions = list(engine.predict(array, text_det_box_thresh=thresh))
        detections = []
        for item in predictions:
            payload = _paddle_payload(item)
            texts = payload.get('rec_texts', []) or []
            scores = payload.get('rec_scores', []) or []
            boxes = payload.get('rec_polys', payload.get('dt_polys', [])) or []
            for i,text in enumerate(texts):
                detections.append(dict(text=str(text), confidence=float(scores[i]) if i<len(scores) else None,
                    box=np.asarray(boxes[i]).tolist() if i<len(boxes) else None))
        return detections, time.perf_counter()-started
    except Exception:
        errors += 1
        raise  # Conservative: stop even on the first error; never repeat completed work.

def evaluate(row, method='original'):
    array = load_common_input(row.image_path, max_long_side=512)
    if method == 'autocontrast':
        array = np.asarray(ImageOps.autocontrast(Image.fromarray(array)))
    elif method == 'unsharp':
        array = np.asarray(Image.fromarray(array).filter(ImageFilter.UnsharpMask(radius=1.5, percent=150, threshold=3)))
    detections, first_time = infer(array, threshold)
    text = ' | '.join(d['text'] for d in detections)
    fallback = selected != '07' and not candidates(text)
    fallback_time = 0.0
    if fallback:
        second, fallback_time = infer(array, 0.7)
        detections = detections + second
        text = ' | '.join(d['text'] for d in detections)
    return dict(file_name=row.file_name, image_id=row.image_id, final_date=row.final_date,
        method=method, detected_text=text, date_recalled=date_recalled(text,row),
        date_candidates_json=json.dumps(candidates(text)), inference_seconds=first_time+fallback_time,
        first_pass_seconds=first_time, fallback_seconds=fallback_time, fallback=fallback,
        detection_count=len(detections), raw_detections_json=json.dumps(detections,ensure_ascii=False),
        text_det_box_thresh=threshold, fallback_threshold=0.7, resolution=512, batch_size=6,cpu_threads=4,
        source='new_inference', model_name='korean_PP-OCRv5_mobile_rec', det_model_name='PP-OCRv5_mobile_det')

results = []
for name in base.index:
    first = saved[selected].loc[name]
    fallback = selected != '07' and not candidates(first.detected_text)
    second = base.loc[name]
    text = first.detected_text + (' | '+second.detected_text if fallback else '')
    results.append(dict(file_name=name, image_id=first.image_id, final_date=first.final_date, method='original',
        detected_text=text, date_recalled=bool(first.date_recalled or (fallback and second.date_recalled)),
        date_candidates_json=json.dumps(candidates(text)),
        inference_seconds=float(first.inference_seconds)+(float(second.inference_seconds) if fallback else 0),
        first_pass_seconds=float(first.inference_seconds), fallback_seconds=float(second.inference_seconds) if fallback else 0,
        fallback=fallback, detection_count=int(first.detection_count)+(int(second.detection_count) if fallback else 0),
        raw_detections_json=first.raw_detections_json, text_det_box_thresh=threshold, resolution=512,
        batch_size=6,cpu_threads=4,source='reused_saved_csv'))

def summary(frame):
    scored=frame.date_recalled.notna()
    return dict(images=len(frame), scored_dates=int(scored.sum()), recalled_dates=int(frame.loc[scored,'date_recalled'].astype(bool).sum()),
        date_token_recall=float(frame.loc[scored,'date_recalled'].astype(bool).mean()),
        mean_seconds=float(frame.inference_seconds.mean()), median_seconds=float(frame.inference_seconds.median()),
        total_detection_boxes=int(frame.detection_count.sum()), mean_detection_boxes=float(frame.detection_count.mean()),
        estimated_3352_minutes=float(frame.inference_seconds.mean()*3352/60), selected_strategy=selected,
        reused_images=int(frame.source.eq('reused_saved_csv').sum()), new_images=int(frame.source.eq('new_inference').sum()))

def run_range(start,end):
    for _, row in fixed.iloc[start:end].iterrows():
        record=evaluate(row)
        results.append(record)
        save(pd.DataFrame([record]), 'new_'+row.file_name.rsplit('.',1)[0])
        print('EVALUATED',len(results),row.file_name,record['date_recalled'],record['inference_seconds'],flush=True)

run_range(10,30)
frame30=pd.DataFrame(results)
save(frame30,'results30')
stats30=summary(frame30)
save(pd.DataFrame([stats30]),'summary30')
print('SUMMARY30',stats30,flush=True)
# Conservative stability criterion: 100% recall on all scorable dates and no execution errors.
stable=stats30['date_token_recall']==1.0 and errors==0
if stable and time.monotonic()<DEADLINE-120:
    run_range(30,50)
    final=pd.DataFrame(results)
    save(final,'results50')
    save(pd.DataFrame([summary(final)]),'summary50')
    print('SUMMARY50',summary(final),flush=True)
else:
    final=frame30

failures=final.loc[final.date_recalled.eq(False)].copy()
failures['failure_reason']=failures.detected_text.map(lambda t: 'valid_date_candidate_but_label_not_recalled' if candidates(t) else 'no_valid_full_date_candidate_in_ocr')
save(failures,'failures')
# Try at most two transformations ONLY on failed images; do not promote based on oracle selection.
preprocess=[]
for method in ['autocontrast','unsharp']:
    for name in failures.file_name:
        if time.monotonic()>=DEADLINE-30:
            break
        row=fixed.loc[fixed.file_name.eq(name)].iloc[0]
        record=evaluate(row,method)
        preprocess.append(record)
        save(pd.DataFrame([record]),method+'_'+name.rsplit('.',1)[0])
if preprocess:
    save(pd.DataFrame(preprocess),'failure_preprocessing')
print('FINAL',summary(final),'FAILURES',failures[['file_name','failure_reason']].to_dict('records'),flush=True)
print('Preprocessing trials:',len(preprocess), 'Not applied to successful images.',flush=True)
del engine
gc.collect()


SIMULATION strategy  recall  total_seconds  mean_seconds  estimated_3352_minutes  retries                                              retry_files  candidate
      07     1.0      13.405282      1.340528               74.890841        0                                                       []      False
     075     1.0      13.382228      1.338223               74.762045        1                                           ["000099.jpg"]       True
      08     1.0      13.284304      1.328430               74.214978        4 ["000058.jpg", "000060.jpg", "000075.jpg", "000099.jpg"]       True SELECTED 08


EVALUATED 11 001097.jpg False 3.1066407000034815


EVALUATED 12 003006.jpg True 0.4554733000040869


EVALUATED 13 000634.jpg True 0.588330099999439


EVALUATED 14 002113.jpg True 0.31934560000081547


...

EVALUATED 29 002728.jpg False 3.0602115000074264


EVALUATED 30 002963.jpg True 0.45357420000073034


SUMMARY30 {'images': 30, 'scored_dates': 29, 'recalled_dates': 23, 'date_token_recall': 0.7931034482758621, 'mean_seconds': 1.2657189833326508, 'median_seconds': 0.8890822999983357, 'total_detection_boxes': 300, 'mean_detection_boxes': 10.0, 'estimated_3352_minutes': 70.71150053551742, 'selected_strategy': '08', 'reused_images': 10, 'new_images': 20}


Preprocessing trials: 12 Not applied to successful images.


125

In [1]:
# Final audit of saved validation only; no OCR execution.
from pathlib import Path
import json
import pandas as pd
OUT=(next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'predict.ipynb').is_file() and (p / 'data').is_dir())) / 'data/validation/ocr_comparison'
RUN="bounded_seed42_validation"
results=pd.read_csv(OUT/f"{RUN}_results30.csv")
prep=pd.read_csv(OUT/f"{RUN}_failure_preprocessing.csv")
notes={
"001097.jpg": "Date printed on a small slanted packet region. OCR misses/garbles date digits; sharpening yields wrong 2021.05.1. Region size/perspective suspected; no geometric trial performed.",
"000227.jpg": "Dot-matrix date on curved yellow bottle label. Date absent from OCR; small size/contrast after 512px reduction suspected.",
"000447.jpg": "Faint date under reflective plastic. Original OCR empty; unsharp alone recovers 2025.11.22. Candidate parser also misses mixed dot/space separators.",
"000978.jpg": "Rotated package with small pale multi-line dates. OCR lacks correct date; rotation/layout and low contrast suspected; not experimentally isolated.",
"000157.jpg": "Small dot-matrix date on curved bottle neck. OCR empty; neither tested transform recovers it. MM/DD versus DD/MM is also ambiguous.",
"002728.jpg": "OCR correctly reads 04 NOV 2021. Numeric-only scoring and candidate detector do not support English month names: evaluation/parser limitation, not a missing OCR date.",
}
review=pd.DataFrame([dict(file_name=name,reviewed_reason=reason) for name,reason in notes.items()])
review.to_csv(OUT/f"{RUN}_failure_review.csv",index=False,encoding='utf-8-sig',mode='x')
summary=[]
for method,g in prep.groupby('method'):
    ok=g.date_recalled.astype(str).str.lower().eq('true')
    summary.append(dict(method=method,tested_failed_images=len(g),recovered_images=int(ok.sum()),
        recovered_files=json.dumps(g.loc[ok,'file_name'].tolist()),additional_seconds=float(g.inference_seconds.sum()),
        deployment_decision='not_promoted_failure_selected_test_only'))
pd.DataFrame(summary).to_csv(OUT/f"{RUN}_preprocessing_summary.csv",index=False,encoding='utf-8-sig',mode='x')
recommendation=pd.DataFrame([dict(
    recommended_strategy='single_threshold_07_provisional',resolution=512,batch_size=6,cpu_threads=4,
    text_det_box_thresh=0.7,det_model='PP-OCRv5_mobile_det',rec_model='korean_PP-OCRv5_mobile_rec',
    adaptive_status='not_promoted_tiny_simulated_gain_and_unstable_30_image_recall',
    adaptive_30_scored=29,adaptive_30_recalled=23,adaptive_30_mean_seconds=float(results.inference_seconds.mean()),
    baseline_07_30_status='not_measured_do_not_attribute_adaptive_metrics_to_baseline',
    expansion50='not_run_30_image_recall_below_100_percent',
    limitation='10 cached plus 20 new images; excludes model initialization, preprocessing and orchestration time; dates are screening recall, not final expiry accuracy',
    next_owner='Date parser: English months, mixed separators, date ordering, multiple manufacture/expiry dates, and candidate confidence; preserve OCR raw data',
)])
recommendation.to_csv(OUT/f"{RUN}_recommendation.csv",index=False,encoding='utf-8-sig',mode='x')
schema=[
('file_name','string','Stable original filename'),
('detections','list[{text,confidence,box}]','Keep raw text and quadrilateral for every OCR item; do not discard non-date text'),
('input_width,input_height','integers','Coordinates refer to EXIF-corrected resized image; retain scale mapping to original'),
('pass_id,threshold,preprocessing','per-pass metadata','Distinguish initial OCR from fallback; avoid treating duplicate boxes as unique detections'),
('date_candidates','list','Derived candidates with source text/box and interpretation; never use labels to trigger fallback'),
('first_pass_seconds,fallback_seconds,total_seconds','floats','Total includes both OCR passes'),
('date_recalled,final_date','evaluation only','Exclude ground-truth fields from production decision inputs'),
]
pd.DataFrame(schema,columns=['field','type','meaning']).to_csv(OUT/f"{RUN}_handoff_schema.csv",index=False,encoding='utf-8-sig',mode='x')
print('Preprocessing:',summary)
print('30-image fallback count:',results.fallback.astype(str).str.lower().eq('true').sum())
print('Final recommendation: retain single 0.7 provisionally; 30-image metrics belong to adaptive 0.8->0.7 only.')
print('No 50-image run. No further OCR. One partial-date label excluded from recall denominator.')


30-image fallback count: 14
Final recommendation: retain single 0.7 provisionally; 30-image metrics belong to adaptive 0.8->0.7 only.
No 50-image run. No further OCR. One partial-date label excluded from recall denominator.


In [1]:
# Complete the fixed seed42 30-image SINGLE threshold=0.7 baseline.
# Reuse 10 standalone records and 10 original-input fallback passes. Only infer missing images.
from pathlib import Path
import os,json,re,time,gc
import numpy as np
import pandas as pd
from PIL import Image,ImageOps
ROOT=next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'predict.ipynb').is_file() and (p / 'data').is_dir())
OUT=ROOT/'data/validation/ocr_comparison'
RUN='single07_seed42_30'
assert not list(OUT.glob(RUN+'*.csv')), 'Existing completion artifacts; inspect instead of rerunning'
MAX_LONG_SIDE=512
CPU_THREADS=4
subset=pd.read_csv(OUT/'bounded_seed42_validation_fixed_subset50.csv',dtype={'image_id':str}).iloc[:30].copy()
assert len(subset)==30 and subset.file_name.is_unique
old07=pd.read_csv(OUT/'paddle_resolution_trial_512_korean_PP-OCRv5_mobile_rec_batch6_boxthresh07_results.csv',dtype={'image_id':str})
adaptive=pd.read_csv(OUT/'bounded_seed42_validation_results30.csv',dtype={'image_id':str})
assert old07.text_det_box_thresh.eq(0.7).all()
records={}
provenance=[]
for _,row in old07.iterrows():
    record=row.to_dict()
    record.update(source='saved_standalone_07',method='original',text_recognition_batch_size=6,cpu_threads=4)
    records[row.file_name]=record
    provenance.append(dict(file_name=row.file_name,source='saved_standalone_07',rerun=False))
for _,row in adaptive.loc[adaptive.source.eq('new_inference') & adaptive.fallback.eq(True)].iterrows():
    # Stored order is first-pass detections + second-pass detections.
    # Higher threshold boxes are a subset of lower threshold boxes. Verify full polygon identity,
    # unique boxes within each pass, and subset membership before recovering the split.
    combined=json.loads(row.raw_detections_json)
    keys=[json.dumps(d['box']) for d in combined]
    split=len(keys)-len(set(keys))
    assert len(set(keys[:split]))==split
    assert len(set(keys[split:]))==len(keys[split:])
    assert set(keys[:split]).issubset(set(keys[split:]))
    detections=combined[split:]
    assert row.file_name not in records
    records[row.file_name]=dict(file_name=row.file_name,image_id=row.image_id,final_date=row.final_date,
        detected_text=' | '.join(d['text'] for d in detections),inference_seconds=float(row.fallback_seconds),
        detection_count=len(detections),raw_detections_json=json.dumps(detections,ensure_ascii=False),
        text_det_box_thresh=0.7,resolution=512,text_recognition_batch_size=6,cpu_threads=4,
        model_name='korean_PP-OCRv5_mobile_rec',det_model_name='PP-OCRv5_mobile_det',
        source='saved_adaptive_07_pass',method='original')
    provenance.append(dict(file_name=row.file_name,source='saved_adaptive_07_pass',rerun=False,
        combined_boxes=len(combined),first_pass_boxes=split,recovered_07_boxes=len(detections)))
missing=subset.loc[~subset.file_name.isin(records)].copy()
assert len(records)==20 and len(missing)==10
print('Reused:',len(records),'Missing:',missing.file_name.tolist(),flush=True)
image_map={p.name:p for p in (ROOT/'data/validation/label_images').rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png'}}
missing['image_path']=missing.file_name.map(image_map)
assert missing.image_path.notna().all()
os.environ['PADDLE_PDX_CACHE_HOME']=str(ROOT/'weights/paddlex')
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK']='1'
os.environ['OMP_NUM_THREADS']='4'
os.environ['MKL_NUM_THREADS']='4'
def load_common_input(path: Path, max_long_side: int = MAX_LONG_SIDE) -> np.ndarray:
    """두 엔진에 동일하게 전달할 EXIF 보정 RGB uint8 배열을 만듭니다."""
    with Image.open(path) as src:
        image = ImageOps.exif_transpose(src).convert("RGB")
    long_side = max(image.size)
    if long_side > max_long_side:
        scale = max_long_side / long_side
        new_size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        image = image.resize(new_size, Image.Resampling.LANCZOS)
    return np.asarray(image)

def normalize_digits(value) -> str:
    return re.sub(r"[^0-9]", "", str(value))

def expected_date_variants(row) -> set[str]:
    """완전한 Y/M/D 라벨을 흔한 YMD, DMY, MDY 숫자열 후보로 변환합니다."""
    if str(row["final_date"]).strip().upper() in {"", "NONE", "NAN"}:
        return set()
    try:
        year, month, day = int(row["year"]), int(row["month"]), int(row["day"])
    except (TypeError, ValueError):
        return set()
    y4, y2 = f"{year:04d}", f"{year % 100:02d}"
    mm, dd = f"{month:02d}", f"{day:02d}"
    m, d = str(month), str(day)
    return {
        y4 + mm + dd, y2 + mm + dd, y4 + m + d, y2 + m + d,
        dd + mm + y4, dd + mm + y2, d + m + y4, d + m + y2,
        mm + dd + y4, mm + dd + y2, m + d + y4, m + d + y2,
    }

def date_recalled(text: str, row):
    variants = expected_date_variants(row)
    if not variants:
        return pd.NA
    digits = normalize_digits(text)
    return any(candidate in digits for candidate in variants)

def _paddle_payload(item):
    payload = getattr(item, "json", item)
    if callable(payload):
        payload = payload()
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    return payload if isinstance(payload, dict) else {}
from paddleocr import PaddleOCR
det_dir=ROOT/'weights/paddlex/official_models/PP-OCRv5_mobile_det'
rec_dir=ROOT/'weights/paddlex/official_models/korean_PP-OCRv5_mobile_rec'
assert det_dir.is_dir() and rec_dir.is_dir()
engine=PaddleOCR(lang='korean',text_detection_model_name='PP-OCRv5_mobile_det',
    text_recognition_model_name='korean_PP-OCRv5_mobile_rec',text_detection_model_dir=str(det_dir),
    text_recognition_model_dir=str(rec_dir),text_recognition_batch_size=6,text_det_limit_side_len=512,
    text_det_limit_type='max',device='cpu',use_doc_orientation_classify=False,use_doc_unwarping=False,
    use_textline_orientation=False,enable_mkldnn=True,cpu_threads=4)
for _,row in missing.iterrows():
    array=load_common_input(row.image_path,max_long_side=512)
    started=time.perf_counter()
    predictions=list(engine.predict(array,text_det_box_thresh=0.7))
    detections=[]
    for item in predictions:
        payload=_paddle_payload(item)
        texts=payload.get('rec_texts',[]) or []
        scores=payload.get('rec_scores',[]) or []
        boxes=payload.get('rec_polys',payload.get('dt_polys',[])) or []
        for i,text in enumerate(texts):
            detections.append(dict(text=str(text),confidence=float(scores[i]) if i<len(scores) else None,
                box=np.asarray(boxes[i]).tolist() if i<len(boxes) else None))
    elapsed=time.perf_counter()-started
    record=dict(file_name=row.file_name,image_id=row.image_id,final_date=row.final_date,
        detected_text=' | '.join(d['text'] for d in detections),inference_seconds=elapsed,
        detection_count=len(detections),raw_detections_json=json.dumps(detections,ensure_ascii=False),
        input_height=array.shape[0],input_width=array.shape[1],text_det_box_thresh=0.7,resolution=512,
        text_recognition_batch_size=6,cpu_threads=4,model_name='korean_PP-OCRv5_mobile_rec',
        det_model_name='PP-OCRv5_mobile_det',source='new_single07',method='original')
    records[row.file_name]=record
    pd.DataFrame([record]).to_csv(OUT/f"{RUN}_new_{row.image_id}.csv",index=False,encoding='utf-8-sig',mode='x')
    provenance.append(dict(file_name=row.file_name,source='new_single07',rerun=False))
    print('NEW',row.file_name,elapsed,flush=True)
del engine
gc.collect()
result=pd.DataFrame([records[name] for name in subset.file_name])
label_by_name=subset.set_index('file_name')
result['date_recalled']=[date_recalled(row.detected_text,label_by_name.loc[row.file_name]) for _,row in result.iterrows()]
result['failure_type']=''
result.loc[result.date_recalled.eq(False),'failure_type']='ocr_date_not_recovered'
nov=result.file_name.eq('002728.jpg')
assert result.loc[nov,'detected_text'].str.contains('04 NOV 2021',regex=False).all()
result.loc[nov,'failure_type']='parser_evaluator_unsupported_english_month'
result['handoff']=''
result.loc[nov,'handoff']='이수민: OCR reads 04 NOV 2021 correctly; English month interpretation/evaluator support required. No new parser implemented.'
scored=result.date_recalled.notna()
summary=dict(images=30,scored_complete_dates=int(scored.sum()),recalled_dates=int(result.loc[scored,'date_recalled'].astype(bool).sum()),
    date_token_recall=float(result.loc[scored,'date_recalled'].astype(bool).mean()),
    mean_seconds=float(result.inference_seconds.mean()),median_seconds=float(result.inference_seconds.median()),
    total_seconds=float(result.inference_seconds.sum()),total_detection_boxes=int(result.detection_count.sum()),
    mean_detection_boxes=float(result.detection_count.mean()),estimated_3352_minutes=float(result.inference_seconds.mean()*3352/60),
    reused_standalone=10,reused_fallback_passes=10,new_images=10,
    ocr_failures=int(result.failure_type.eq('ocr_date_not_recovered').sum()),
    parser_evaluator_cases=int(result.failure_type.eq('parser_evaluator_unsupported_english_month').sum()),
    recommendation='retain_single_threshold07_provisional_with_documented_30_image_limitations',
    resolution=512,text_det_box_thresh=0.7,batch_size=6,cpu_threads=4,
    timing_note='Mix of saved standalone, saved warm fallback, and newly measured single-pass timings; not a fresh all-30 benchmark')
for frame,suffix in [(result,'results'),(pd.DataFrame([summary]),'summary'),
    (result.loc[result.failure_type.ne('')],'failures'),(pd.DataFrame(provenance),'provenance')]:
    frame.to_csv(OUT/f"{RUN}_{suffix}.csv",index=False,encoding='utf-8-sig',mode='x')
print('SUMMARY',summary,flush=True)
print(result.loc[result.failure_type.ne(''),['file_name','final_date','detected_text','failure_type']].to_string(index=False))


Reused: 20 Missing: ['003006.jpg', '000634.jpg', '002113.jpg', '001500.jpg', '001427.jpg', '001780.jpg', '003201.jpg', '003238.jpg', '002622.jpg', '002963.jpg']


NEW 003006.jpg 0.9762588000012329


NEW 000634.jpg 0.5580463000005693


NEW 002113.jpg 0.6338221000041813


NEW 001500.jpg 0.47581930000160355


...

NEW 003201.jpg 3.162621499999659


NEW 003238.jpg 1.7013283999986015


NEW 002622.jpg 0.7651037000032375


NEW 002963.jpg 0.9827638999995543


file_name final_date                                                                                                                                                                                                                   detected_text                               failure_type
001097.jpg 2021-06-11                                                            D4 | 영양정보 | 링당 | 자트를 | 650 mg | 기준치에 | 한수화를 | 619 | 1E9 | 15 | 지방 | 18 | 트랜스지방 | 41% | 포화지방 | 7 | 부터 | 레스데를 | 69 | 40% | 5 mg 미닌 | 1% | 기한 | 08:15 A04F5 | 9g | 16 %                     ocr_date_not_recovered
000227.jpg 2028-09-17                                                                                                                        48900m | 영양정보 |  | 사농중관 |  0 | MA | Cre E0 | 3 |  | 지 | 경프NE |  | 2012503214Sm | 사오수기 |  | 의 | 17 | VD44                     ocr_date_not_recovered
000447.jpg 2025-11-22                                                                                                                 

## 최종 권고 보완: 고정 30장 threshold=0.7 단독 검증
512px / PP-OCRv5_mobile_det / korean_PP-OCRv5_mobile_rec / batch6 / CPU threads=4 / threshold=0.7을 잠정 기준으로 유지합니다. 기존 10장 단독 결과와 적응형 실행에서 분리·검증한 0.7 재시도 10장을 재사용하고, 결과가 없는 10장만 새로 실행했습니다.
완전 라벨 29장 중 숫자 기반 날짜 토큰 회수 22장(75.86%). 평균 1.316726초, 중앙값 1.019570초, 총 39.501791초. 박스 총 379개, 평균 12.633개. 3,352장 환산 73.5611분. 별도 실행 시점 및 warm fallback 시간을 재사용한 합산이며 새 30장 일괄 벤치마크가 아닙니다.
OCR 날짜 미회수: 001097, 000227, 000447, 000978, 000645, 000157. 000645는 2026-05-13을 2026-15-13으로 오인식했습니다.
002728.jpg의 04 NOV 2021은 OCR 인식 성공이나 숫자 기반 evaluator 미지원입니다. 이수민 담당자에게 영문 월 해석 및 평가기 지원 사항으로 인계합니다. 기존 재현율 지표를 임의로 변경하거나 영문 월 파서를 추가하지 않았습니다.
이 수치는 이전 권고의 0.7 단독 30장 미검증 항목을 대체합니다. 최종 제출 수준의 정확도는 확보되지 않았으며, 적응형 지표와 혼동하지 않습니다. 보호 파일 수정·추가 최적화·전처리·50장/300장 실행은 하지 않았습니다.
